In [ ]:
# Validate the explicitly named population shapefile.
import os
from pathlib import Path

DID_DIR = Path("data/public/population")
DID_SHP = DID_DIR / "population_districts_2015.shp"

if not DID_SHP.exists():
    raise FileNotFoundError(
        f"Required population shapefile not found: {DID_SHP}"
    )


In [ ]:
# Load and summarize the population and urban-park spatial layers.
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.ops import unary_union

DID_SHP = r"data/public/population/population_districts_2015.shp"
BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"

print("exists DID :", os.path.exists(DID_SHP), DID_SHP)
print("exists BASE:", os.path.exists(BASE_SHP), BASE_SHP)

def read_did_robust(path: str) -> gpd.GeoDataFrame:
    """
    Try to read DID shapefile robustly:
    1) try pyogrio (often reads datetimes as strings)
    2) try fiona ignoring the problematic datetime field A16_011
    3) try fiona reading only needed fields (exclude A16_011)
    """

    try:
        did0 = gpd.read_file(path, engine="pyogrio")
        return did0
    except Exception as e1:
        print("read DID with pyogrio failed ->", repr(e1))

    try:
        did1 = gpd.read_file(path, engine="fiona", ignore_fields=["A16_011"])
        return did1
    except TypeError as e_ignore_not_supported:

        print("fiona ignore_fields not supported ->", repr(e_ignore_not_supported))
    except Exception as e2:
        print("read DID with fiona(ignore_fields) failed ->", repr(e2))

    cols = ["A16_001", "A16_002", "A16_003", "A16_004", "A16_005", "A16_006", "A16_007", "A16_008", "A16_009", "A16_010"]
    try:
        did2 = gpd.read_file(path, engine="fiona", columns=cols)
        return did2
    except Exception as e3:
        raise RuntimeError(f"Failed to read DID shp with all methods. Last error: {e3}") from e3

did = read_did_robust(DID_SHP)
base = gpd.read_file(BASE_SHP)

print("\n=== DID loaded ===")
print("rows:", len(did))
print("crs :", did.crs)
print("columns:", list(did.columns))

print("\n=== BASE loaded ===")
print("rows:", len(base))
print("crs :", base.crs)

if did.crs is None:
    did = did.set_crs("EPSG:6668")
    print("WARNING: DID CRS None -> set to EPSG:6668")
if base.crs is None:
    base = base.set_crs("EPSG:4326")
    print("WARNING: BASE CRS None -> set to EPSG:4326")

bad = (~did.is_valid).sum()
if bad > 0:
    print(f"invalid DID geometries: {bad} -> buffer(0) fix")
    did["geometry"] = did.buffer(0)

POP_COL = "A16_005"
AREA_COL = "A16_006"

if POP_COL not in did.columns or AREA_COL not in did.columns:
    raise KeyError(f"Missing required fields in DID. Need {POP_COL},{AREA_COL}. Got: {list(did.columns)}")

did[POP_COL] = pd.to_numeric(did[POP_COL], errors="coerce")
did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")
did["density_person_per_km2"] = did[POP_COL] / did[AREA_COL]

print("\n=== Missing ===")
print("POP missing ratio :", did[POP_COL].isna().mean())
print("AREA missing ratio:", did[AREA_COL].isna().mean())

print("\n=== DID summary ===")
print("Total pop:", float(np.nansum(did[POP_COL])))
print("Total area (km2):", float(np.nansum(did[AREA_COL])))

def quantiles(s):
    s = pd.Series(s).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {}
    return s.quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 1]).to_dict()

print("Pop quantiles:", quantiles(did[POP_COL]))
print("Area(km2) quantiles:", quantiles(did[AREA_COL]))
print("Density(person/km2) quantiles:", quantiles(did["density_person_per_km2"]))

base_outline = base.dissolve()

did_plot = did.to_crs(base_outline.crs) if str(did.crs) != str(base_outline.crs) else did

fig, ax = plt.subplots(figsize=(10, 7))
base_outline.boundary.plot(ax=ax, color="0.5", linewidth=1.2, zorder=1)
did_plot.plot(ax=ax, color="tab:blue", alpha=0.25, linewidth=0, zorder=2)
ax.set_title("Study area boundary (gray) + DID polygons (blue)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

try:
    utm = base_outline.estimate_utm_crs()
    if utm is None:
        raise ValueError("estimate_utm_crs returned None")
except Exception as e:
    print("WARNING: estimate_utm_crs failed -> fallback to EPSG:3857. Err:", repr(e))
    utm = "EPSG:3857"

base_m = base_outline.to_crs(utm)
did_m = did_plot.to_crs(utm)

base_geom = unary_union(base_m.geometry)
did_geom  = unary_union(did_m.geometry)

base_area = float(base_geom.area)
inter_area = float(base_geom.intersection(did_geom).area)
coverage_ratio = inter_area / base_area if base_area > 0 else np.nan

print("\n=== Coverage (area-based, sanity only) ===")
print("base area (m2):", base_area)
print("intersection area (m2):", inter_area)
print("area coverage ratio (DID ∩ base)/base:", coverage_ratio)

print("\n=== DID head (key fields) ===")
try:
    display(did[[POP_COL, AREA_COL, "density_person_per_km2"]].head())
except NameError:
    print(did[[POP_COL, AREA_COL, "density_person_per_km2"]].head())


In [ ]:
# Clip the population layer to the study boundary.
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.ops import unary_union

DID_SHP = r"data/public/population/population_districts_2015.shp"
BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"

POP_COL, AREA_COL = "A16_005", "A16_006"

did = gpd.read_file(DID_SHP, engine="fiona", ignore_fields=["A16_011"])
base = gpd.read_file(BASE_SHP)

if did.crs is None:
    did = did.set_crs("EPSG:6668")
if base.crs is None:
    base = base.set_crs("EPSG:4326")

bad = (~did.is_valid).sum()
if bad > 0:
    did["geometry"] = did.buffer(0)

did[POP_COL]  = pd.to_numeric(did[POP_COL], errors="coerce")
did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")

base_outline = base.dissolve()

did2 = did.to_crs(base_outline.crs) if str(did.crs) != str(base_outline.crs) else did.copy()

utm = base_outline.estimate_utm_crs()
if utm is None:
    utm = "EPSG:3857"
base_m = base_outline.to_crs(utm)
did_m  = did2.to_crs(utm)

base_geom = unary_union(base_m.geometry)

did_area = did_m.geometry.area
inter_geom = did_m.geometry.intersection(base_geom)
inter_area = inter_geom.area

ratio = np.where(did_area > 0, inter_area / did_area, 0.0)

m = inter_area > 0
did_clip = did_m.loc[m].copy()
did_clip["inter_area_m2"] = inter_area[m]
did_clip["area_m2"] = did_area[m]
did_clip["ratio_in_base"] = ratio[m]

did_clip["pop_in_base"] = did_clip[POP_COL].astype(float) * did_clip["ratio_in_base"]
did_clip["area_km2_in_base_attr"] = did_clip[AREA_COL].astype(float) * did_clip["ratio_in_base"]
did_clip["density_person_per_km2_in_base_attr"] = did_clip["pop_in_base"] / did_clip["area_km2_in_base_attr"].replace(0, np.nan)

print("=== Clipped DID to study area ===")
print("DID rows (original):", len(did_m))
print("DID rows (intersect base):", len(did_clip))
print("Sum pop_in_base:", float(did_clip["pop_in_base"].sum()))
print("Sum area_in_base (attr km2):", float(did_clip["area_km2_in_base_attr"].sum()))

base_area = float(base_geom.area)
inter_total_area = float(did_clip["inter_area_m2"].sum())
print("Area coverage ratio (DID ∩ base)/base:", inter_total_area / base_area if base_area > 0 else np.nan)

did_clip_plot = did_clip.to_crs(base_outline.crs)
fig, ax = plt.subplots(figsize=(10, 7))
base_outline.boundary.plot(ax=ax, color="0.5", linewidth=1.2, zorder=1)
did_clip_plot.plot(ax=ax, color="tab:blue", alpha=0.25, linewidth=0, zorder=2)
ax.set_title("Clipped DID (blue) within Study Area boundary (gray)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# Report spatial bounds for coordinate-reference-system checks.
print("DID bounds:", did.total_bounds)
print("BASE bounds:", base.total_bounds)


In [ ]:
# Reconstruct the clipped population layer from the source datasets.
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union

DID_SHP = r"data/public/population/population_districts_2015.shp"
BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"

POP_COL, AREA_COL = "A16_005", "A16_006"

if "did_clip" not in globals():
    print("did_clip not found -> recomputing did_clip ...")
    did = gpd.read_file(DID_SHP, engine="fiona", ignore_fields=["A16_011"])
    base = gpd.read_file(BASE_SHP)

    if did.crs is None:
        did = did.set_crs("EPSG:6668")
    if base.crs is None:
        base = base.set_crs("EPSG:4326")

    bad = (~did.is_valid).sum()
    if bad > 0:
        did["geometry"] = did.buffer(0)

    did[POP_COL]  = pd.to_numeric(did[POP_COL], errors="coerce")
    did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")

    base_outline = base.dissolve()
    did2 = did.to_crs(base_outline.crs) if str(did.crs) != str(base_outline.crs) else did.copy()

    utm = base_outline.estimate_utm_crs()
    if utm is None:
        utm = "EPSG:3857"

    base_m = base_outline.to_crs(utm)
    did_m  = did2.to_crs(utm)

    base_geom = unary_union(base_m.geometry)

    did_area = did_m.geometry.area
    inter_geom = did_m.geometry.intersection(base_geom)
    inter_area = inter_geom.area

    ratio = np.where(did_area > 0, inter_area / did_area, 0.0)
    m = inter_area > 0

    did_clip = did_m.loc[m].copy()
    did_clip["inter_area_m2"] = inter_area[m]
    did_clip["area_m2"] = did_area[m]
    did_clip["ratio_in_base"] = ratio[m]
    did_clip["pop_in_base"] = did_clip[POP_COL].astype(float) * did_clip["ratio_in_base"]
    did_clip["area_km2_in_base_attr"] = did_clip[AREA_COL].astype(float) * did_clip["ratio_in_base"]

    print("did_clip rows:", len(did_clip))
    print("Sum pop_in_base:", float(did_clip["pop_in_base"].sum()))

did_clip = did_clip.copy()
did_clip = did_clip[did_clip["pop_in_base"] > 0].copy()

did_clip["pop_pt"] = did_clip.geometry.representative_point()
pop_gdf = gpd.GeoDataFrame(
    did_clip[["pop_in_base", "area_km2_in_base_attr"]].copy(),
    geometry=did_clip["pop_pt"],
    crs=did_clip.crs,
)

print("pop_gdf rows:", len(pop_gdf))
print("pop total:", float(pop_gdf["pop_in_base"].sum()))
print("pop CRS:", pop_gdf.crs)


In [ ]:
# Calculate effective served population for each park.
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union

OUT_DIR = r"outputs"
OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_table.csv")

BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"

DID_SHP = r"data/public/population/population_districts_2015.shp"
POP_COL, AREA_COL = "A16_005", "A16_006"

if "pop_gdf" not in globals():
    print("pop_gdf not found -> recompute from DID (clip to base) ...")
    did = gpd.read_file(DID_SHP, engine="fiona", ignore_fields=["A16_011"])
    base = gpd.read_file(BASE_SHP)

    if did.crs is None:
        did = did.set_crs("EPSG:6668")
    if base.crs is None:
        base = base.set_crs("EPSG:4326")

    bad = (~did.is_valid).sum()
    if bad > 0:
        did["geometry"] = did.buffer(0)

    did[POP_COL]  = pd.to_numeric(did[POP_COL], errors="coerce")
    did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")

    base_outline = base.dissolve()
    did2 = did.to_crs(base_outline.crs) if str(did.crs) != str(base_outline.crs) else did.copy()

    utm0 = base_outline.estimate_utm_crs()
    if utm0 is None:
        utm0 = "EPSG:3857"

    base_m = base_outline.to_crs(utm0)
    did_m  = did2.to_crs(utm0)

    base_geom = unary_union(base_m.geometry)

    did_area = did_m.geometry.area
    inter_geom = did_m.geometry.intersection(base_geom)
    inter_area = inter_geom.area

    # Population is allocated to the study area in proportion to each district's intersected area.
    ratio = np.where(did_area > 0, inter_area / did_area, 0.0)
    m = inter_area > 0

    did_clip = did_m.loc[m].copy()
    did_clip["ratio_in_base"] = ratio[m]
    did_clip["pop_in_base"] = did_clip[POP_COL].astype(float) * did_clip["ratio_in_base"]
    did_clip["area_km2_in_base_attr"] = did_clip[AREA_COL].astype(float) * did_clip["ratio_in_base"]

    did_clip = did_clip[did_clip["pop_in_base"] > 0].copy()
    did_clip["pop_pt"] = did_clip.geometry.representative_point()

    pop_gdf = gpd.GeoDataFrame(
        did_clip[["pop_in_base", "area_km2_in_base_attr"]].copy(),
        geometry=did_clip["pop_pt"],
        crs=did_clip.crs,
    )

    print("pop_gdf rows:", len(pop_gdf))
    print("pop total:", float(pop_gdf["pop_in_base"].sum()))
    print("pop CRS:", pop_gdf.crs)

if "parks_gdf" not in globals():
    print("parks_gdf not found -> loading from:", OUT_TABLE_CSV)
    dfp = pd.read_csv(OUT_TABLE_CSV, encoding="utf-8-sig")

    need_cols = ["osm_id_norm","Lat","Lng","area_m2","park_class_name","land_cost_yen","annual_maint_yen","annual_benefit_yen"]
    miss = [c for c in need_cols if c not in dfp.columns]
    if miss:
        raise KeyError(f"park_payback_table.csv missing columns: {miss}\nCurrent columns={list(dfp.columns)}")

    TYPE_MAP = {
        "Block Park": "A",
        "Neighborhood Park": "B",
        "District Park": "C",
        "Comprehensive Park": "D",
        "Regional Park": "E",
    }
    dfp["abcde"] = dfp["park_class_name"].astype(str).str.strip().map(TYPE_MAP)

    parks_gdf = gpd.GeoDataFrame(
        dfp,
        geometry=gpd.points_from_xy(dfp["Lng"], dfp["Lat"]),
        crs="EPSG:4326",
    )

    print("parks rows:", len(parks_gdf))
    print("parks CRS:", parks_gdf.crs)
    print("type counts:\n", parks_gdf["abcde"].value_counts(dropna=False))

RADIUS_M = 6000.0
GAMMA_AREA = 0.5

LOGSIG = {"A": 11.7571, "B": 11.9702, "C": 12.3210, "D": 12.2487, "E": 12.7412}
BETA   = {"A": -0.4160, "B": -0.4193, "C": -0.4217, "D": -0.3829, "E": -0.4308}

base_outline = gpd.read_file(BASE_SHP).dissolve()
utm = base_outline.estimate_utm_crs()
if utm is None:
    utm = "EPSG:3857"
print("Using metric CRS:", utm)

parks_m = parks_gdf.to_crs(utm)
pop_m   = pop_gdf.to_crs(utm)

park_xy = np.c_[parks_m.geometry.x.values, parks_m.geometry.y.values].astype(np.float64)
pop_xy  = np.c_[pop_m.geometry.x.values,   pop_m.geometry.y.values].astype(np.float64)
pop_w   = pop_m["pop_in_base"].values.astype(np.float64)

park_type = parks_m["abcde"].values
beta_arr = np.array([BETA.get(t, np.nan) for t in park_type], dtype=np.float64)

logsig_vals = np.array([LOGSIG.get(t, np.nan) for t in park_type], dtype=np.float64)
logsig_mean = np.nanmean(list(LOGSIG.values()))
omega = np.exp(logsig_vals - logsig_mean)

area_m2 = pd.to_numeric(parks_m["area_m2"], errors="coerce").fillna(0).values.astype(np.float64)
area_m2 = np.clip(area_m2, 1.0, None)
attr = omega * (area_m2 ** GAMMA_AREA)

try:
    from scipy.spatial import cKDTree
    tree = cKDTree(park_xy)
    neigh = tree.query_ball_point(pop_xy, r=RADIUS_M)
    print("Using scipy.spatial.cKDTree")
except Exception as e:
    print("scipy cKDTree not available, fallback to sklearn BallTree. Err:", repr(e))
    from sklearn.neighbors import BallTree
    tree = BallTree(park_xy, metric="euclidean")
    neigh = tree.query_radius(pop_xy, r=RADIUS_M, return_distance=False)
    neigh = [list(x) for x in neigh]
    print("Using sklearn BallTree")

E_pop = np.zeros(len(parks_m), dtype=np.float64)
covered_pop = 0.0

for c_idx, idxs in enumerate(neigh):
    if len(idxs) == 0:
        continue
    idxs = np.asarray(idxs, dtype=int)

    dx = park_xy[idxs, 0] - pop_xy[c_idx, 0]
    dy = park_xy[idxs, 1] - pop_xy[c_idx, 1]
    dist = np.sqrt(dx*dx + dy*dy)
    dist = np.maximum(dist, 1.0)

    b = beta_arr[idxs]
    a = attr[idxs]
    w = a * (dist ** b)

    sw = w.sum()
    if not np.isfinite(sw) or sw <= 0:
        continue

    # Each population point is distributed across nearby parks by normalized attractiveness and distance-decay weights.
    s = w / sw
    contrib = pop_w[c_idx] * s

    np.add.at(E_pop, idxs, contrib)
    covered_pop += pop_w[c_idx]

parks_out = parks_gdf.copy()
parks_out["E_pop"] = E_pop
parks_out["E_pop_per_m2"] = parks_out["E_pop"] / parks_out["area_m2"].replace(0, np.nan)

print("Total pop (DID in base):", float(pop_w.sum()))
print("Total pop covered by >=1 park within 6km:", float(covered_pop))
print("Sum E_pop across parks:", float(np.nansum(E_pop)))

OUT_EPOP_CSV = os.path.join(OUT_DIR, "park_exposure_Epop_alltypes.csv")
parks_out.to_csv(OUT_EPOP_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_EPOP_CSV)

ab = parks_out[parks_out["abcde"].isin(["A","B"])].copy()
print("\nA/B E_pop summary:")
print(ab["E_pop"].describe(percentiles=[0.1,0.5,0.9,0.99]))


In [ ]:
# Recalculate effective served population using the finalized spatial specification.
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union

OUT_DIR = r"outputs"
OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_table.csv")

BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"
DID_SHP  = r"data/public/population/population_districts_2015.shp"

POP_COL, AREA_COL = "A16_005", "A16_006"

base = gpd.read_file(BASE_SHP)
if base.crs is None:
    base = base.set_crs("EPSG:4326")
base_outline = base.dissolve()

utm = base_outline.estimate_utm_crs()
if utm is None:
    utm = "EPSG:3857"
print("Using metric CRS:", utm)

did = gpd.read_file(DID_SHP, engine="fiona", ignore_fields=["A16_011"])

if did.crs is None:
    did = did.set_crs("EPSG:4326", allow_override=True)

bad = (~did.is_valid).sum()
if bad > 0:
    did["geometry"] = did.buffer(0)

did[POP_COL]  = pd.to_numeric(did[POP_COL], errors="coerce")
did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")

did_m  = did.to_crs(utm)
base_m = base_outline.to_crs(utm)
base_geom = unary_union(base_m.geometry)

did_area = did_m.geometry.area
inter_geom = did_m.geometry.intersection(base_geom)
inter_area = inter_geom.area

ratio = np.where(did_area > 0, inter_area / did_area, 0.0)
m = inter_area > 0

did_clip_ll = did.loc[m].copy()
did_clip_ll["ratio_in_base"] = ratio[m]
did_clip_ll["pop_in_base"] = did_clip_ll[POP_COL].astype(float) * did_clip_ll["ratio_in_base"]
did_clip_ll["area_km2_in_base_attr"] = did_clip_ll[AREA_COL].astype(float) * did_clip_ll["ratio_in_base"]
did_clip_ll = did_clip_ll[did_clip_ll["pop_in_base"] > 0].copy()

did_clip_ll["pop_pt"] = did_clip_ll.geometry.representative_point()
pop_gdf_ll = gpd.GeoDataFrame(
    did_clip_ll[["pop_in_base", "area_km2_in_base_attr"]].copy(),
    geometry=did_clip_ll["pop_pt"],
    crs="EPSG:4326",
)

print("\nPop points rebuilt:")
print("pop_gdf_ll rows:", len(pop_gdf_ll))
print("pop total:", float(pop_gdf_ll["pop_in_base"].sum()))
print("pop bounds (lon/lat):", pop_gdf_ll.total_bounds)

dfp = pd.read_csv(OUT_TABLE_CSV, encoding="utf-8-sig")

name = dfp["park_class_name"].astype(str).str.strip().str.lower()
MAP_LOWER = {
    "block park": "A",
    "neighborhood park": "B",
    "district park": "C",
    "comprehensive park": "D",
    "regional park": "E",
}
dfp["abcde"] = name.map(MAP_LOWER)

dfp["Lng"] = pd.to_numeric(dfp["Lng"], errors="coerce")
dfp["Lat"] = pd.to_numeric(dfp["Lat"], errors="coerce")
m_ok = dfp["Lng"].between(-180, 180) & dfp["Lat"].between(-90, 90)
dfp = dfp[m_ok].copy()

parks_gdf = gpd.GeoDataFrame(
    dfp,
    geometry=gpd.points_from_xy(dfp["Lng"], dfp["Lat"]),
    crs="EPSG:4326",
)

print("\nParks loaded:")
print("parks rows:", len(parks_gdf))
print("parks bounds (lon/lat):", parks_gdf.total_bounds)
print("type counts:\n", parks_gdf["abcde"].value_counts(dropna=False).head(10))

minx, miny, maxx, maxy = base_outline.total_bounds
parks_gdf = parks_gdf[parks_gdf["Lng"].between(minx-0.2, maxx+0.2) & parks_gdf["Lat"].between(miny-0.2, maxy+0.2)].copy()
print("parks rows after bbox filter:", len(parks_gdf))

RADIUS_M = 6000.0
GAMMA_AREA = 0.5

LOGSIG = {"A": 11.7571, "B": 11.9702, "C": 12.3210, "D": 12.2487, "E": 12.7412}
BETA   = {"A": -0.4160, "B": -0.4193, "C": -0.4217, "D": -0.3829, "E": -0.4308}

parks_m = parks_gdf.to_crs(utm)
pop_m   = pop_gdf_ll.to_crs(utm)

print("\nMeter bounds check:")
print("parks_m bounds:", parks_m.total_bounds)
print("pop_m   bounds:", pop_m.total_bounds)

park_xy = np.c_[parks_m.geometry.x.values, parks_m.geometry.y.values].astype(np.float64)
pop_xy  = np.c_[pop_m.geometry.x.values,   pop_m.geometry.y.values].astype(np.float64)
pop_w   = pop_m["pop_in_base"].values.astype(np.float64)

park_type = parks_m["abcde"].values
beta_arr = np.array([BETA.get(t, np.nan) for t in park_type], dtype=np.float64)

logsig_vals = np.array([LOGSIG.get(t, np.nan) for t in park_type], dtype=np.float64)
logsig_mean = np.nanmean(list(LOGSIG.values()))
omega = np.exp(np.where(np.isfinite(logsig_vals), logsig_vals, logsig_mean) - logsig_mean)

area_m2 = pd.to_numeric(parks_m["area_m2"], errors="coerce").fillna(0).values.astype(np.float64)
area_m2 = np.clip(area_m2, 1.0, None)
attr = omega * (area_m2 ** GAMMA_AREA)

from scipy.spatial import cKDTree
tree = cKDTree(park_xy)

dmin, _ = tree.query(pop_xy, k=1)
print("\nNearest park distance to each pop point (m):")
print(pd.Series(dmin).describe(percentiles=[0.1,0.5,0.9,0.99]))

neigh = tree.query_ball_point(pop_xy, r=RADIUS_M)

E_pop = np.zeros(len(parks_m), dtype=np.float64)
covered_pop = 0.0
n_nonempty = 0

for c_idx, idxs in enumerate(neigh):
    if len(idxs) == 0:
        continue
    n_nonempty += 1
    idxs = np.asarray(idxs, dtype=int)

    dx = park_xy[idxs, 0] - pop_xy[c_idx, 0]
    dy = park_xy[idxs, 1] - pop_xy[c_idx, 1]
    dist = np.sqrt(dx*dx + dy*dy)
    dist = np.maximum(dist, 1.0)

    b = beta_arr[idxs]
    b = np.where(np.isfinite(b), b, BETA["B"])
    a = attr[idxs]

    w = a * (dist ** b)
    sw = w.sum()
    if not np.isfinite(sw) or sw <= 0:
        continue

    s = w / sw
    np.add.at(E_pop, idxs, pop_w[c_idx] * s)
    covered_pop += pop_w[c_idx]

parks_out = parks_gdf.copy()
parks_out["E_pop"] = E_pop

print("\nCoverage results:")
print("Total pop (DID in base):", float(pop_w.sum()))
print("Total pop covered by >=1 park within 6km:", float(covered_pop))
print("Share covered:", float(covered_pop / pop_w.sum()))
print("pop points with >=1 park within 6km:", n_nonempty, "/", len(pop_w))
print("Sum E_pop across parks:", float(np.nansum(E_pop)))

OUT_EPOP_CSV = os.path.join(OUT_DIR, "park_exposure_Epop_alltypes.csv")
parks_out.to_csv(OUT_EPOP_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_EPOP_CSV)

print("\nA/B E_pop summary:")
print(parks_out.loc[parks_out["abcde"].isin(["A","B"]), "E_pop"].describe(percentiles=[0.1,0.5,0.9,0.99]))


In [ ]:
# Inspect the distribution of effective served population.
import numpy as np
import pandas as pd

dmin, _ = tree.query(pop_xy, k=1)
within6 = (dmin <= 6000.0)

covered_pop_w = float(np.sum(pop_w[within6]))
total_pop_w   = float(np.sum(pop_w))

print("Nearest distance (m) summary:")
print(pd.Series(dmin).describe(percentiles=[0.1,0.5,0.9,0.99]))

print("\nCoverage within 6km (pop-weighted):")
print("covered_pop =", covered_pop_w)
print("total_pop   =", total_pop_w)
print("share       =", covered_pop_w / total_pop_w)

print("\nCoverage within 6km (point count):")
print("covered points:", int(within6.sum()), "/", len(within6), "=", float(within6.mean()))


In [ ]:
# Standardize park-type labels for the planning-stage analysis.
import pandas as pd
import numpy as np

if "dfp" not in globals():
    import os
    OUT_DIR = r"outputs"
    OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_table.csv")
    dfp = pd.read_csv(OUT_TABLE_CSV, encoding="utf-8-sig")

s_raw = dfp["park_class_name"].astype(str).str.strip()
s = s_raw.str.lower()

def classify_one(raw, low):

    if "block" in low: return "A"
    if "neigh" in low: return "B"
    if "district" in low: return "C"
    if "comprehensive" in low: return "D"
    if "regional" in low: return "E"

    if "街区" in raw or "街区公園" in raw: return "A"
    if "近隣" in raw or "近隣公園" in raw: return "B"
    if "地区" in raw or "地区公園" in raw: return "C"
    if "総合" in raw or "総合公園" in raw: return "D"
    if "広域" in raw or "広域公園" in raw: return "E"
    return np.nan

abc = [classify_one(r, l) for r, l in zip(s_raw.values, s.values)]
dfp["abcde_fix"] = abc

print("type counts (fixed):\n", pd.Series(dfp["abcde_fix"]).value_counts(dropna=False))

unknown = dfp.loc[pd.isna(dfp["abcde_fix"]), "park_class_name"].astype(str).str.strip()
print("\nUnknown park_class_name top 30:")
print(unknown.value_counts().head(30))


In [ ]:
# Generate the finalized park-level effective-population table.
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from scipy.spatial import cKDTree

OUT_DIR = r"outputs"
OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_table.csv")

BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"
DID_SHP  = r"data/public/population/population_districts_2015.shp"
POP_COL, AREA_COL = "A16_005", "A16_006"

base = gpd.read_file(BASE_SHP)
if base.crs is None:
    base = base.set_crs("EPSG:4326")
base_outline = base.dissolve()
utm = base_outline.estimate_utm_crs() or "EPSG:3857"
print("Using metric CRS:", utm)

did = gpd.read_file(DID_SHP, engine="fiona", ignore_fields=["A16_011"])
if did.crs is None:

    did = did.set_crs("EPSG:4326", allow_override=True)

bad = (~did.is_valid).sum()
if bad > 0:
    did["geometry"] = did.buffer(0)

did[POP_COL]  = pd.to_numeric(did[POP_COL], errors="coerce")
did[AREA_COL] = pd.to_numeric(did[AREA_COL], errors="coerce")

did_m  = did.to_crs(utm)
base_m = base_outline.to_crs(utm)
base_geom = unary_union(base_m.geometry)

did_area = did_m.geometry.area
inter_area = did_m.geometry.intersection(base_geom).area
ratio = np.where(did_area > 0, inter_area / did_area, 0.0)

m = inter_area > 0
did_clip_ll = did.loc[m].copy()
did_clip_ll["pop_in_base"] = did_clip_ll[POP_COL].astype(float) * ratio[m]
did_clip_ll = did_clip_ll[did_clip_ll["pop_in_base"] > 0].copy()

did_clip_ll["pop_pt"] = did_clip_ll.geometry.representative_point()
pop_gdf_ll = gpd.GeoDataFrame(
    did_clip_ll[["pop_in_base"]].copy(),
    geometry=did_clip_ll["pop_pt"],
    crs="EPSG:4326",
)

print("pop points:", len(pop_gdf_ll), "| pop total:", float(pop_gdf_ll["pop_in_base"].sum()))

dfp = pd.read_csv(OUT_TABLE_CSV, encoding="utf-8-sig")
dfp["Lng"] = pd.to_numeric(dfp["Lng"], errors="coerce")
dfp["Lat"] = pd.to_numeric(dfp["Lat"], errors="coerce")
dfp = dfp[dfp["Lng"].between(-180, 180) & dfp["Lat"].between(-90, 90)].copy()

s_raw = dfp["park_class_name"].astype(str).str.strip()
s = s_raw.str.lower()

def classify_one(raw, low):
    if "block" in low: return "A"
    if "neigh" in low: return "B"
    if "district" in low: return "C"
    if "comprehensive" in low: return "D"
    if "regional" in low: return "E"
    if "街区" in raw or "街区公園" in raw: return "A"
    if "近隣" in raw or "近隣公園" in raw: return "B"
    if "地区" in raw or "地区公園" in raw: return "C"
    if "総合" in raw or "総合公園" in raw: return "D"
    if "広域" in raw or "広域公園" in raw: return "E"
    return np.nan

dfp["abcde"] = [classify_one(r, l) for r, l in zip(s_raw.values, s.values)]
print("type counts:\n", dfp["abcde"].value_counts(dropna=False))

parks_gdf = gpd.GeoDataFrame(
    dfp,
    geometry=gpd.points_from_xy(dfp["Lng"], dfp["Lat"]),
    crs="EPSG:4326",
)

RADIUS_M = 6000.0
GAMMA_AREA = 0.5

LOGSIG = {"A": 11.7571, "B": 11.9702, "C": 12.3210, "D": 12.2487, "E": 12.7412}
BETA   = {"A": -0.4160, "B": -0.4193, "C": -0.4217, "D": -0.3829, "E": -0.4308}

parks_m = parks_gdf.to_crs(utm)
pop_m   = pop_gdf_ll.to_crs(utm)

park_xy = np.c_[parks_m.geometry.x.values, parks_m.geometry.y.values].astype(np.float64)
pop_xy  = np.c_[pop_m.geometry.x.values,   pop_m.geometry.y.values].astype(np.float64)
pop_w   = pop_m["pop_in_base"].values.astype(np.float64)

park_type = parks_m["abcde"].values
beta_arr = np.array([BETA.get(t, np.nan) for t in park_type], dtype=np.float64)
beta_arr = np.where(np.isfinite(beta_arr), beta_arr, BETA["B"])

logsig_vals = np.array([LOGSIG.get(t, np.nan) for t in park_type], dtype=np.float64)
logsig_mean = np.nanmean(list(LOGSIG.values()))
omega = np.exp(np.where(np.isfinite(logsig_vals), logsig_vals, logsig_mean) - logsig_mean)

area_m2 = pd.to_numeric(parks_m["area_m2"], errors="coerce").fillna(0).values.astype(np.float64)
area_m2 = np.clip(area_m2, 1.0, None)
attr = omega * (area_m2 ** GAMMA_AREA)

tree = cKDTree(park_xy)

dmin, _ = tree.query(pop_xy, k=1)
print("\nNearest park distance (m) quantiles:",
      pd.Series(dmin).quantile([0,0.1,0.5,0.9,0.99,1]).to_dict())

neigh = tree.query_ball_point(pop_xy, r=RADIUS_M)

E_pop = np.zeros(len(parks_m), dtype=np.float64)
covered_pop = 0.0
n_nonempty = 0

for c_idx, idxs in enumerate(neigh):
    if len(idxs) == 0:
        continue
    n_nonempty += 1
    idxs = np.asarray(idxs, dtype=int)

    dx = park_xy[idxs, 0] - pop_xy[c_idx, 0]
    dy = park_xy[idxs, 1] - pop_xy[c_idx, 1]
    dist = np.sqrt(dx*dx + dy*dy)
    dist = np.maximum(dist, 1.0)

    w = attr[idxs] * (dist ** beta_arr[idxs])
    sw = w.sum()
    if not np.isfinite(sw) or sw <= 0:
        continue

    # E_pop conserves covered population by normalizing park-attraction weights at each population point.
    s_share = w / sw
    np.add.at(E_pop, idxs, pop_w[c_idx] * s_share)
    covered_pop += pop_w[c_idx]

parks_out = parks_gdf.copy()
parks_out["E_pop"] = E_pop

print("\nCoverage within 6km (pop-weighted):")
print("covered_pop =", float(covered_pop))
print("total_pop   =", float(pop_w.sum()))
print("share       =", float(covered_pop / pop_w.sum()))
print("covered pop-points:", n_nonempty, "/", len(pop_w))
print("Sum E_pop across parks:", float(np.nansum(E_pop)))

OUT_EPOP_CSV = os.path.join(OUT_DIR, "park_exposure_Epop_alltypes.csv")
parks_out.to_csv(OUT_EPOP_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_EPOP_CSV)

print("\nA/B E_pop summary:")
print(parks_out.loc[parks_out["abcde"].isin(["A","B"]), "E_pop"].describe(percentiles=[0.1,0.5,0.9,0.99]))


In [ ]:
# Inspect the finalized park-level variables and category coverage.
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

ab = parks_out.loc[parks_out["abcde"].isin(["A","B"]), "E_pop"].copy()
print(ab.describe(percentiles=[0.01,0.1,0.25,0.5,0.75,0.9,0.99,0.999]))


In [ ]:
# Fit initial park-type-specific benefit models.
import numpy as np
import pandas as pd

d = parks_out.copy()

ycol = "payback_years" if "payback_years" in d.columns else "payback_hat_years"
print("Using y =", ycol)

d["land_price_yen_per_m2"] = d["land_cost_yen"] / d["area_m2"].replace(0, np.nan)
d["maint_yen_per_m2"] = d["annual_maint_yen"] / d["area_m2"].replace(0, np.nan)

X_cols = ["land_price_yen_per_m2", "area_m2", "E_pop", "maint_yen_per_m2"]

d_ab = d[d["abcde"].isin(["A","B"])].copy()
d_ab[ycol] = pd.to_numeric(d_ab[ycol], errors="coerce")
m = np.isfinite(d_ab[ycol]) & (d_ab[ycol] > 0)
d_ab = d_ab[m].copy()

for c in X_cols:
    d_ab[c] = pd.to_numeric(d_ab[c], errors="coerce")
    d_ab[f"ln_{c}"] = np.log(d_ab[c].clip(lower=1e-9))
d_ab["ln_y"] = np.log(d_ab[ycol].clip(lower=1e-9))

def ols_numpy(X, y):
    n = X.shape[0]
    X1 = np.c_[np.ones(n), X]
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    y_hat = X1 @ beta
    resid = y - y_hat
    ssr = np.sum(resid**2)
    sst = np.sum((y - y.mean())**2)
    r2 = 1 - ssr/sst if sst > 0 else np.nan

    k1 = X1.shape[1]
    dof = max(n - k1, 1)
    sigma2 = ssr / dof
    XtX_inv = np.linalg.inv(X1.T @ X1)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    return beta, se, r2, n

def fit_one(t):
    sub = d_ab[d_ab["abcde"] == t].dropna(subset=[f"ln_{c}" for c in X_cols] + ["ln_y"]).copy()
    if len(sub) < 200:
        print(f"\nType {t}: too few samples =", len(sub))
        return None

    X = sub[[f"ln_{c}" for c in X_cols]].values
    y = sub["ln_y"].values
    beta, se, r2, n = ols_numpy(X, y)

    print(f"\n=== Type {t} log-linear OLS (numpy) ===")
    print("n =", n, "| R2 =", float(r2))
    print("ln(T) = a + Σ b_j ln(x_j)")
    print(f"a = {beta[0]:.6f}  (se={se[0]:.6f})")
    for j, col in enumerate(X_cols, start=1):
        print(f"b_{col} = {beta[j]:.6f}  (se={se[j]:.6f})")

    A0 = float(np.exp(beta[0]))
    print("\nMultiplicative form:")
    print(f"T ≈ {A0:.6g}"
          f" * land_price^{beta[1]:.4f}"
          f" * area^{beta[2]:.4f}"
          f" * E_pop^{beta[3]:.4f}"
          f" * maint^{beta[4]:.4f}")

    return beta, r2, n

betaA = fit_one("A")
betaB = fit_one("B")


In [ ]:
# Fit log-linear benefit models with smearing correction.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear.csv")

PAYBACK_HORIZON = 50.0
EPS = 1e-9

d = parks_out.copy()

d["land_price_yen_per_m2"] = d["land_cost_yen"] / d["area_m2"].replace(0, np.nan)
d["maint_yen_per_m2"]      = d["annual_maint_yen"] / d["area_m2"].replace(0, np.nan)

X_cols = ["land_price_yen_per_m2", "area_m2", "E_pop", "maint_yen_per_m2"]
y_col  = "annual_benefit_yen"
y_pay  = "payback_years" if "payback_years" in d.columns else None

for c in X_cols + [y_col, "annual_maint_yen", "land_cost_yen", "area_m2", "E_pop"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

def ols_numpy(X, y):
    n = X.shape[0]
    X1 = np.c_[np.ones(n), X]
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    yhat = X1 @ beta
    resid = y - yhat
    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = 1 - ssr/sst if sst > 0 else np.nan
    return beta, resid, r2, n

def fit_with_smearing(type_char):
    sub = d[d["abcde"] == type_char].copy()

    m = np.isfinite(sub[y_col]) & (sub[y_col] > 0)
    for c in X_cols:
        m &= np.isfinite(sub[c]) & (sub[c] > 0)

    sub = sub[m].copy()
    if len(sub) < 200:
        print(f"Type {type_char}: too few samples = {len(sub)}")
        return None

    y = np.log(sub[y_col].clip(lower=EPS)).values
    X = np.column_stack([np.log(sub[c].clip(lower=EPS)).values for c in X_cols])

    beta, resid, r2, n = ols_numpy(X, y)
    # Duan's smearing factor corrects retransformation bias from the log-linear model.
    smear = float(np.mean(np.exp(resid)))

    print(f"\n=== Type {type_char} benefit model + smearing ===")
    print("n =", n, "| R2 =", float(r2))
    print("smearing S =", smear)
    print("ln(benefit) = a + Σ b ln(x)")
    print("a =", float(beta[0]))
    for j, col in enumerate(X_cols, start=1):
        print(f"b_{col} =", float(beta[j]))

    A0 = float(np.exp(beta[0]) * smear)
    print("\nMultiplicative (smear-corrected) form:")
    print(f"benefit_hat ≈ {A0:.6g}"
          f" * land_price^{beta[1]:.4f}"
          f" * area^{beta[2]:.4f}"
          f" * E_pop^{beta[3]:.4f}"
          f" * maint^{beta[4]:.4f}")

    return beta, smear

def predict_benefit_smear(type_char, beta, smear):
    idx = (d["abcde"] == type_char)
    sub = d.loc[idx, :].copy()

    m = np.ones(len(sub), dtype=bool)
    for c in X_cols:
        m &= np.isfinite(sub[c].values) & (sub[c].values > 0)

    Xp = np.column_stack([np.log(sub.loc[m, c].clip(lower=EPS)).values for c in X_cols])
    ln_hat = beta[0] + Xp @ beta[1:]
    # Predictions are returned to the monetary scale with the fitted smearing correction.
    bhat = np.exp(ln_hat) * smear

    out = np.full(len(sub), np.nan, dtype=float)
    out[m] = bhat
    return idx, out

resA = fit_with_smearing("A")
resB = fit_with_smearing("B")

d["benefit_hat_2step_smear_yen"] = np.nan
if resA is not None:
    betaA, smearA = resA
    idxA, outA = predict_benefit_smear("A", betaA, smearA)
    d.loc[idxA, "benefit_hat_2step_smear_yen"] = outA
if resB is not None:
    betaB, smearB = resB
    idxB, outB = predict_benefit_smear("B", betaB, smearB)
    d.loc[idxB, "benefit_hat_2step_smear_yen"] = outB

net_hat = d["benefit_hat_2step_smear_yen"] - d["annual_maint_yen"]
d["payback_hat_2step_smear_years"] = np.where(
    d["abcde"].isin(["A","B"]) & np.isfinite(net_hat) & (net_hat > 0) & np.isfinite(d["land_cost_yen"]),
    d["land_cost_yen"] / net_hat,
    np.nan
)

ab = d[d["abcde"].isin(["A","B"])].copy()
ab.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved:", OUT_CSV)

print("\n=== Predicted payback (smear-corrected) summary (A/B) ===")
print(ab["payback_hat_2step_smear_years"].describe(percentiles=[0.1,0.5,0.9,0.99]))

m_valid = np.isfinite(ab["payback_hat_2step_smear_years"])
print("\nvalid n:", int(m_valid.sum()), "/", len(ab))
if m_valid.any():
    print("<=50 share (pred):", float((ab.loc[m_valid, "payback_hat_2step_smear_years"] <= PAYBACK_HORIZON).mean()))

if y_pay is not None and y_pay in ab.columns:
    ab[y_pay] = pd.to_numeric(ab[y_pay], errors="coerce")
    m_obs = np.isfinite(ab[y_pay]) & (ab[y_pay] > 0)
    print("\n=== Observed payback summary (A/B) ===")
    print(ab.loc[m_obs, y_pay].describe(percentiles=[0.1,0.5,0.9,0.99]))
    if m_obs.any():
        print("<=50 share (obs):", float((ab.loc[m_obs, y_pay] <= PAYBACK_HORIZON).mean()))

m2 = np.isfinite(ab["benefit_hat_2step_smear_yen"]) & (ab[y_col] > 0)
if m2.any():
    ratio = (ab.loc[m2, "benefit_hat_2step_smear_yen"] / ab.loc[m2, y_col]).replace([np.inf, -np.inf], np.nan).dropna()
    print("\nMedian(b_hat / b_obs) =", float(ratio.median()))
    print("P10/P90(b_hat / b_obs) =", float(ratio.quantile(0.1)), "/", float(ratio.quantile(0.9)))


In [ ]:
# Calibrate predicted payback classifications to observed outcomes.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear.csv")
OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_paybackcal.csv")

H = 50.0
EPS = 1e-12

ab = pd.read_csv(IN_CSV, encoding="utf-8-sig")

need = ["abcde","payback_years","benefit_hat_2step_smear_yen","annual_maint_yen","land_cost_yen"]
miss = [c for c in need if c not in ab.columns]
if miss:
    raise KeyError(f"Missing columns in {IN_CSV}: {miss}")

for c in ["payback_years","benefit_hat_2step_smear_yen","annual_maint_yen","land_cost_yen"]:
    ab[c] = pd.to_numeric(ab[c], errors="coerce")

base_mask = (
    ab["abcde"].isin(["A","B"]) &
    np.isfinite(ab["payback_years"]) & (ab["payback_years"] > 0) &
    np.isfinite(ab["benefit_hat_2step_smear_yen"]) & (ab["benefit_hat_2step_smear_yen"] > 0) &
    np.isfinite(ab["annual_maint_yen"]) &
    np.isfinite(ab["land_cost_yen"]) & (ab["land_cost_yen"] > 0)
)

def pred_share_leq50(sub_df, s):
    """Return the predicted share with payback at or below 50 years for scale ``s``. Non-positive net benefits imply infinite payback, and the calibration sample remains fixed across scale values."""
    b = sub_df["benefit_hat_2step_smear_yen"].values
    m = sub_df["annual_maint_yen"].values
    lc = sub_df["land_cost_yen"].values

    net = s * b - m
    T = np.full_like(net, np.inf, dtype=float)
    ok = net > EPS
    T[ok] = lc[ok] / net[ok]
    return float(np.mean(T <= H))

# This calibration matches the observed prevalence of recovery within 50 years, not individual payback times.
def find_s_for_target(sub_df, target, s_lo=0.05, s_hi=5.0, iters=50):
    """
    Use bisection to match the predicted within-50-year share to the observed
    target. The share is monotonic because larger benefit scales shorten payback.
    """

    f_lo = pred_share_leq50(sub_df, s_lo)
    f_hi = pred_share_leq50(sub_df, s_hi)

    if target <= f_lo:
        return s_lo, f_lo, f_hi
    if target >= f_hi:
        return s_hi, f_lo, f_hi

    lo, hi = s_lo, s_hi
    for _ in range(iters):
        mid = (lo + hi) / 2.0
        f_mid = pred_share_leq50(sub_df, mid)
        if f_mid < target:
            lo = mid
        else:
            hi = mid
    s_star = (lo + hi) / 2.0
    return s_star, f_lo, f_hi

scales = {}
for t in ["A","B"]:
    sub = ab[base_mask & (ab["abcde"] == t)].copy()
    obs_share = float(np.mean(sub["payback_years"].values <= H))
    s_star, f_lo, f_hi = find_s_for_target(sub, obs_share)

    pred_star = pred_share_leq50(sub, s_star)
    scales[t] = s_star

    print(f"\nType {t}:")
    print("n =", len(sub))
    print("obs <=50 share =", obs_share)
    print("pred <=50 share @s_lo =", f_lo, "| @s_hi =", f_hi)
    print("calibrated s =", s_star)
    print("pred <=50 share @s* =", pred_star)

ab["s_paybackcal"] = ab["abcde"].map(scales)
net = ab["s_paybackcal"] * ab["benefit_hat_2step_smear_yen"] - ab["annual_maint_yen"]
ab["payback_hat_paybackcal_years"] = np.where(net > EPS, ab["land_cost_yen"] / net, np.inf)

sub_all = ab[base_mask].copy()
obs_all = float(np.mean(sub_all["payback_years"] <= H))
pred_all = float(np.mean(sub_all["payback_hat_paybackcal_years"] <= H))

print("\n=== Overall (A+B) ===")
print("obs <=50 share =", obs_all)
print("pred <=50 share (payback-cal) =", pred_all)
print(sub_all["payback_hat_paybackcal_years"].replace(np.inf, np.nan).describe(percentiles=[0.1,0.5,0.9,0.99]))

ab.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved:", OUT_CSV)


In [ ]:
# Apply the calibrated park-type benefit and payback equations.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "park_exposure_Epop_alltypes.csv")
OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_final_formula_payback.csv")

EPS = 1e-12

if "parks_out" in globals():
    df = parks_out.copy()
else:
    df = pd.read_csv(IN_CSV, encoding="utf-8-sig")

need = ["abcde","area_m2","E_pop","land_cost_yen","annual_maint_yen","annual_benefit_yen"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise KeyError(f"Missing columns: {miss}")

for c in ["area_m2","E_pop","land_cost_yen","annual_maint_yen","annual_benefit_yen"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["land_price_yen_per_m2"] = df["land_cost_yen"] / df["area_m2"].replace(0, np.nan)
df["maint_yen_per_m2"]      = df["annual_maint_yen"] / df["area_m2"].replace(0, np.nan)

A_land = 0.8123162063057531
A_area = 0.9646576847394284
A_epop = 0.10729039801485121
A_maint = 0.2502979465109065
A_const_smear = 0.0514607
sA = 0.6100810390009688
A_const_final = A_const_smear * sA

B_land = 0.7719228968398684
B_area = 0.7120351974772099
B_epop = 0.09901200893262105
B_maint = 0.2884708133568651
aB = -1.247133336706259
SB = 1.5726997170833297
B_const_smear = float(np.exp(aB) * SB)
sB = 0.7893653258042048
B_const_final = B_const_smear * sB

print("A_const_smear =", A_const_smear, "| sA =", sA, "| A_const_final =", A_const_final)
print("B_const_smear =", B_const_smear, "| sB =", sB, "| B_const_final =", B_const_final)

df_ab = df[df["abcde"].isin(["A","B"])].copy()

def safe_pow(x, p):
    x = pd.to_numeric(x, errors="coerce")
    return np.power(np.clip(x, EPS, None), p)

df_ab["benefit_hat_final_yen"] = np.nan

mA = df_ab["abcde"] == "A"
mB = df_ab["abcde"] == "B"

df_ab.loc[mA, "benefit_hat_final_yen"] = (
    A_const_final
    * safe_pow(df_ab.loc[mA, "land_price_yen_per_m2"], A_land)
    * safe_pow(df_ab.loc[mA, "area_m2"], A_area)
    * safe_pow(df_ab.loc[mA, "E_pop"], A_epop)
    * safe_pow(df_ab.loc[mA, "maint_yen_per_m2"], A_maint)
)

df_ab.loc[mB, "benefit_hat_final_yen"] = (
    B_const_final
    * safe_pow(df_ab.loc[mB, "land_price_yen_per_m2"], B_land)
    * safe_pow(df_ab.loc[mB, "area_m2"], B_area)
    * safe_pow(df_ab.loc[mB, "E_pop"], B_epop)
    * safe_pow(df_ab.loc[mB, "maint_yen_per_m2"], B_maint)
)

net = df_ab["benefit_hat_final_yen"] - df_ab["annual_maint_yen"]
df_ab["payback_hat_final_years"] = np.where(
    np.isfinite(net) & (net > EPS) & np.isfinite(df_ab["land_cost_yen"]) & (df_ab["land_cost_yen"] > 0),
    df_ab["land_cost_yen"] / net,
    np.inf
)

H = 50.0
m_obs = np.isfinite(df_ab.get("payback_years", np.nan)) & (df_ab.get("payback_years", np.nan) > 0)
m_pred = np.isfinite(df_ab["payback_hat_final_years"]) & (df_ab["payback_hat_final_years"] > 0) & np.isfinite(df_ab["payback_hat_final_years"])

print("\n=== Final predicted payback summary (A/B) ===")
print(pd.Series(df_ab["payback_hat_final_years"]).replace(np.inf, np.nan).describe(percentiles=[0.1,0.5,0.9,0.99]))

if "payback_years" in df_ab.columns and m_obs.any():
    print("\n<=50 share (obs):", float((df_ab.loc[m_obs, "payback_years"] <= H).mean()))
print("<=50 share (pred_final):", float((df_ab.loc[np.isfinite(df_ab["payback_hat_final_years"]), "payback_hat_final_years"] <= H).mean()))

keep_cols = [
    "osm_id", "osm_id_norm", "park_class_name", "abcde",
    "Lat","Lng","area_m2","E_pop",
    "land_cost_yen","annual_maint_yen","annual_benefit_yen",
    "land_price_yen_per_m2","maint_yen_per_m2",
    "benefit_hat_final_yen","payback_hat_final_years"
]
keep_cols = [c for c in keep_cols if c in df_ab.columns]

df_ab[keep_cols].to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved:", OUT_CSV)

print("\n=== Final formulas for Results ===")
print(f"Type A: benefit_hat = {A_const_final:.6g} * land_price^{A_land:.4f} * area^{A_area:.4f} * E_pop^{A_epop:.4f} * maint^{A_maint:.4f}")
print(f"Type B: benefit_hat = {B_const_final:.6g} * land_price^{B_land:.4f} * area^{B_area:.4f} * E_pop^{B_epop:.4f} * maint^{B_maint:.4f}")
print("Payback_hat = land_cost / (benefit_hat - annual_maint)")


In [ ]:
# Calibrate continuous predicted payback against observed payback.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear.csv")
OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal.csv")

EPS = 1e-12
CAP_YEARS = 300.0

ab = pd.read_csv(IN_CSV, encoding="utf-8-sig")

need = ["abcde","payback_years","benefit_hat_2step_smear_yen","annual_maint_yen","land_cost_yen"]
miss = [c for c in need if c not in ab.columns]
if miss:
    raise KeyError(f"Missing columns in {IN_CSV}: {miss}")

for c in ["payback_years","benefit_hat_2step_smear_yen","annual_maint_yen","land_cost_yen"]:
    ab[c] = pd.to_numeric(ab[c], errors="coerce")

base_mask = (
    ab["abcde"].isin(["A","B"]) &
    np.isfinite(ab["payback_years"]) & (ab["payback_years"] > 0) &
    np.isfinite(ab["benefit_hat_2step_smear_yen"]) & (ab["benefit_hat_2step_smear_yen"] > 0) &
    np.isfinite(ab["annual_maint_yen"]) &
    np.isfinite(ab["land_cost_yen"]) & (ab["land_cost_yen"] > 0)
)

def pred_payback(s, b, m, lc):
    """Predict payback years, assigning ``CAP_YEARS`` when net benefit is non-positive."""
    net = s * b - m
    T = np.full_like(net, CAP_YEARS, dtype=float)
    ok = net > EPS
    T[ok] = lc[ok] / net[ok]
    return np.clip(T, EPS, CAP_YEARS)

def loss_log_mae(s, b, m, lc, Tobs):
    """Return median absolute log error, a threshold-independent robust loss."""
    Tpred = pred_payback(s, b, m, lc)
    return float(np.median(np.abs(np.log(Tpred) - np.log(Tobs))))

# Continuous calibration minimizes robust log-scale error in park-level payback years.
def search_s_continuous(sub, s_lo=0.05, s_hi=2.0, n_grid=240, n_refine=160):
    """Use a logarithmic coarse search followed by local refinement for the non-smooth loss."""
    b = sub["benefit_hat_2step_smear_yen"].values
    m = sub["annual_maint_yen"].values
    lc = sub["land_cost_yen"].values
    Tobs = np.clip(sub["payback_years"].values, EPS, CAP_YEARS)

    grid = np.logspace(np.log10(s_lo), np.log10(s_hi), n_grid)
    vals = np.array([loss_log_mae(s, b, m, lc, Tobs) for s in grid])
    i0 = int(np.argmin(vals))
    s0 = float(grid[i0])

    lo = max(s_lo, s0 / 1.35)
    hi = min(s_hi, s0 * 1.35)
    grid2 = np.logspace(np.log10(lo), np.log10(hi), n_refine)
    vals2 = np.array([loss_log_mae(s, b, m, lc, Tobs) for s in grid2])
    i1 = int(np.argmin(vals2))
    s1 = float(grid2[i1])
    return s1, float(vals2[i1]), float(s0), float(vals[i0])

scales = {}
for t in ["A","B"]:
    sub = ab[base_mask & (ab["abcde"] == t)].copy()
    s_star, L_star, s0, L0 = search_s_continuous(sub)
    scales[t] = s_star

    b = sub["benefit_hat_2step_smear_yen"].values
    m = sub["annual_maint_yen"].values
    lc = sub["land_cost_yen"].values
    Tobs = np.clip(sub["payback_years"].values, EPS, CAP_YEARS)
    Tpred = pred_payback(s_star, b, m, lc)

    ratio = (Tpred / Tobs)
    print(f"\nType {t}: n={len(sub)}")
    print(f"best s (continuous) = {s_star:.6f}")
    print(f"log-MAE(median) = {L_star:.6f}")
    print(f"median(Tpred/Tobs) = {float(np.median(ratio)):.4f} | P10/P90 = {float(np.quantile(ratio,0.1)):.4f}/{float(np.quantile(ratio,0.9)):.4f}")

print("\nContinuous-cal scales:", scales)

ab["s_contcal"] = ab["abcde"].map(scales)
ab["benefit_hat_contcal_yen"] = ab["benefit_hat_2step_smear_yen"] * ab["s_contcal"]

net = ab["benefit_hat_contcal_yen"] - ab["annual_maint_yen"]
ab["payback_hat_contcal_years"] = np.where(net > EPS, ab["land_cost_yen"] / net, np.inf)

sub_all = ab[base_mask].copy()
Tobs = np.clip(sub_all["payback_years"].values, EPS, CAP_YEARS)
Tpred = np.clip(sub_all["payback_hat_contcal_years"].replace(np.inf, CAP_YEARS).values, EPS, CAP_YEARS)

print("\n=== Overall (A+B) on observed subset ===")
print("median log-abs-error =", float(np.median(np.abs(np.log(Tpred) - np.log(Tobs)))))
ratio_all = Tpred / Tobs
print("median(Tpred/Tobs) =", float(np.median(ratio_all)))
print("P10/P90(Tpred/Tobs) =", float(np.quantile(ratio_all,0.1)), "/", float(np.quantile(ratio_all,0.9)))
print(pd.Series(Tpred).describe(percentiles=[0.1,0.5,0.9,0.99]))

ab.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved:", OUT_CSV)


In [ ]:
# Compare observed and predicted benefits and payback outcomes.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal.csv")

EPS = 1e-12
CAP_YEARS = 300.0

TITLE_SCALE = 0.82
YTICK_SCALE = 0.82

FONT_SCALE = 1.00
FIGSIZE = (8.27, 3.4)

BOTTOM_FOR_LEGEND = 0.3
RIGHT_FOR_CBAR = 0.035

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11 * FONT_SCALE,
    "axes.titlesize": 12 * FONT_SCALE * TITLE_SCALE,
    "ytick.labelsize": 11 * FONT_SCALE * YTICK_SCALE,
    "xtick.labelsize": 11 * FONT_SCALE * 0.95,

    "axes.labelsize": 11 * FONT_SCALE,
    "legend.fontsize": 10 * FONT_SCALE,
})

BASE_CMAP_NAME = "viridis"
WHITEN = 0.58
DENSITY_ALPHA = 0.95

def make_pastel_cmap(name="viridis", whiten=0.58):
    cmap = plt.get_cmap(name)
    colors = cmap(np.linspace(0, 1, 256))
    colors[:, :3] = (1 - whiten) * colors[:, :3] + whiten * 1.0
    return ListedColormap(colors, name=f"{name}_pastel_w{whiten:.2f}")

PASTEL_CMAP = make_pastel_cmap(BASE_CMAP_NAME, WHITEN)

EFF = [pe.Stroke(linewidth=5.2 * FONT_SCALE, foreground="white", alpha=0.95), pe.Normal()]
LINE_ID = dict(color="#0b3d91", linestyle="--", linewidth=2.6 * FONT_SCALE, alpha=0.98, zorder=7)
LINE_B  = dict(color="#b22222", linestyle=":",  linewidth=2.0 * FONT_SCALE, alpha=0.95, zorder=7)
CAL_LN  = dict(color="#111111", linewidth=2.4 * FONT_SCALE, marker="o",
               markersize=4.6 * FONT_SCALE, alpha=0.95, zorder=8)

def _to_num(s):
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def prep_xy(df, xcol, ycol, cap=None):
    x = _to_num(df[xcol]); y = _to_num(df[ycol])
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[m]; y = y[m]
    if cap is not None:
        x = np.clip(x, EPS, cap)
        y = np.clip(y, EPS, cap)
    return x, y

def quantile_window(lx, ly, qlo=0.01, qhi=0.99, pad=0.05):
    xlo, xhi = np.quantile(lx, qlo), np.quantile(lx, qhi)
    ylo, yhi = np.quantile(ly, qlo), np.quantile(ly, qhi)
    dx = (xhi - xlo) * pad
    dy = (yhi - ylo) * pad
    return (xlo - dx, xhi + dx, ylo - dy, yhi + dy)

def draw_ref_lines(ax, lo, hi):
    xx = np.linspace(lo, hi, 320)
    l1, = ax.plot(xx, xx, **LINE_ID)
    l2, = ax.plot(xx, xx + np.log10(2), **LINE_B)
    l3, = ax.plot(xx, xx - np.log10(2), **LINE_B)
    for ln in (l1, l2, l3):
        ln.set_path_effects(EFF)

def calibration_curve(lx, ly, q=12):
    lx = np.asarray(lx); ly = np.asarray(ly)
    if lx.size < 30:
        return pd.DataFrame({"lx_med": [], "ly_med": []})

    q_eff = int(min(q, max(4, lx.size // 10)))
    bins = pd.qcut(lx, q=q_eff, duplicates="drop")
    cal = (
        pd.DataFrame({"lx": lx, "ly": ly, "bin": bins})
        .groupby("bin")
        .agg(lx_med=("lx","median"), ly_med=("ly","median"), n=("lx","size"))
        .reset_index(drop=True)
        .sort_values("lx_med")
    )
    return cal

def compute_hist2d_counts(lx, ly, bins=65, rng=None):

    H, xedges, yedges = np.histogram2d(lx, ly, bins=bins, range=rng)
    return H

def plot_panel(ax, lx, ly, rng, bins, norm):

    h = ax.hist2d(
        lx, ly,
        bins=bins,
        range=rng,
        norm=norm,
        cmap=PASTEL_CMAP,
        alpha=DENSITY_ALPHA
    )

    xlo, xhi = rng[0]
    ylo, yhi = rng[1]
    lo = max(xlo, ylo)
    hi = min(xhi, yhi)
    draw_ref_lines(ax, lo, hi)

    cal = calibration_curve(lx, ly, q=12)
    if len(cal) > 0:
        ln, = ax.plot(cal["lx_med"], cal["ly_med"], **CAL_LN)
        ln.set_path_effects(EFF)

    ax.set_xlim(rng[0])
    ax.set_ylim(rng[1])
    ax.grid(True, alpha=0.18)
    return h[3]

df = pd.read_csv(IN_CSV, encoding="utf-8-sig")
df = df[df["abcde"].isin(["A","B"])].copy()

must = ["abcde", "payback_years", "payback_hat_contcal_years"]
miss = [c for c in must if c not in df.columns]
if miss:
    raise KeyError(f"Missing columns for payback: {miss}")

has_benefit = ("annual_benefit_yen" in df.columns) and ("benefit_hat_contcal_yen" in df.columns)
if not has_benefit:
    raise KeyError("The four-panel figure requires annual_benefit_yen and benefit_hat_contcal_yen.")

PANELS = [
    ("A", "payback", "payback_hat_contcal_years", "payback_years", CAP_YEARS,
     "Type A payback time", "Predicted (log$_{10}$ years)", "Observed (log$_{10}$ years)"),
    ("A", "benefit", "benefit_hat_contcal_yen", "annual_benefit_yen", None,
     "Type A annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
    ("B", "payback", "payback_hat_contcal_years", "payback_years", CAP_YEARS,
     "Type B payback time", "Predicted (log$_{10}$ years)", "Observed (log$_{10}$ years)"),
    ("B", "benefit", "benefit_hat_contcal_yen", "annual_benefit_yen", None,
     "Type B annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
]

bins = 65

panel_data = []
global_vmax = 1

for (t, kind, xcol, ycol, cap, title, xlabel, ylabel) in PANELS:
    df_t = df[df["abcde"] == t].copy()
    x, y = prep_xy(df_t, xcol, ycol, cap=cap)
    lx = np.log10(x); ly = np.log10(y)
    xlo, xhi, ylo, yhi = quantile_window(lx, ly, 0.01, 0.99, pad=0.05)
    rng = [[xlo, xhi], [ylo, yhi]]

    H = compute_hist2d_counts(lx, ly, bins=bins, rng=rng)
    vmax = int(np.nanmax(H)) if H.size else 1
    global_vmax = max(global_vmax, vmax)

    panel_data.append((lx, ly, rng, title, xlabel, ylabel))

shared_norm = LogNorm(vmin=1, vmax=max(1, global_vmax))

fig, axs = plt.subplots(1, 4, figsize=FIGSIZE, constrained_layout=False)

fig.subplots_adjust(
    left=0.06,
    right=1.0 - RIGHT_FOR_CBAR - 0.02,
    top=0.92,
    bottom=BOTTOM_FOR_LEGEND,
    wspace=0.32
)

last_mappable = None
for i, (ax, (lx, ly, rng, title, xlabel, ylabel)) in enumerate(zip(axs, panel_data)):
    last_mappable = plot_panel(ax, lx, ly, rng, bins=bins, norm=shared_norm)
    ax.set_title(title)

    ax.set_xlabel(xlabel)
    if i == 0:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel("")

cax = fig.add_axes([1.0 - RIGHT_FOR_CBAR, BOTTOM_FOR_LEGEND, 0.012, 0.92 - BOTTOM_FOR_LEGEND])
cb = fig.colorbar(last_mappable, cax=cax)
cb.set_label("Bin count (log scale)")

legend_handles = [
    Line2D([0],[0], **LINE_ID, label="Identity (y = x)"),
    Line2D([0],[0], **LINE_B,  label="×2 / ×0.5 error band"),
    Line2D([0],[0], **CAL_LN,  label="Binned medians"),
]

for h in legend_handles:
    h.set_path_effects(EFF)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.02),
    handlelength=3.0
)

out_png = os.path.join(OUT_DIR, "fig_R3_AB_4panel_row.png")
fig.savefig(out_png, dpi=300, bbox_inches="tight")
print("Saved:", out_png)

plt.show()
plt.close(fig)


In [ ]:
# Evaluate candidate-site payback on the spatial planning grid.
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from matplotlib.colors import Normalize, LinearSegmentedColormap

FONT_SCALE = 2.0
plt.rcParams.update({
    "font.size": 10 * FONT_SCALE,
    "axes.titlesize": 12 * FONT_SCALE,
    "axes.labelsize": 10 * FONT_SCALE,
    "xtick.labelsize": 9 * FONT_SCALE,
    "ytick.labelsize": 9 * FONT_SCALE,
})

BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"
DID_SHP  = r"data/public/population/population_districts_2015.shp"
PARKS_CSV = r"outputs/park_payback_table.csv"
LANDPRICE_SHP = r"data/public/land_prices/official_land_prices.shp"

GRID_STEP_M = 250
R_M = 6000.0
EPS = 1e-9
CAP_YEARS = 300.0

POP_COL = "A16_005"
DID_CRS_FALLBACK = "EPSG:6668"
LP_CRS_FALLBACK  = "EPSG:6668"
LP_VALUE_COL = "L01_008"

A_HA, B_HA = 0.18, 1.10
A_AREA_M2 = A_HA * 10_000.0
B_AREA_M2 = B_HA * 10_000.0

MARKER_EXTRA_SHRINK = 0.20
S_MIN, S_MAX = 1.0, 20.0

Y_5, Y_20, Y_50 = 0.25, 0.45, 0.82

BENEFIT_PARAMS = {
    "A": dict(c=0.0313952, a_lp=0.8123, a_area=0.9647, a_epop=0.1073, a_maint=0.2503),
    "B": dict(c=0.3566980, a_lp=0.7719, a_area=0.7120, a_epop=0.0990, a_maint=0.2885),
}

PARAM_LOGESIGMA_BETA = {
    "A": (11.7571, -0.4160),
    "B": (11.9702, -0.4193),
    "C": (12.3210, -0.4217),
    "D": (12.2487, -0.3829),
    "E": (12.7412, -0.4308),
}
SIGMA = {k: float(np.exp(v[0])) for k, v in PARAM_LOGESIGMA_BETA.items()}
BETA  = {k: float(v[1]) for k, v in PARAM_LOGESIGMA_BETA.items()}

def safe_read_shp(path, columns_needed=None, crs_fallback=None):
    kwargs = {}
    if columns_needed is not None:
        kwargs["columns"] = columns_needed
    try:
        gdf = gpd.read_file(path, engine="pyogrio", **kwargs)
    except Exception as e1:
        print("[safe_read_shp] pyogrio failed -> fallback to fiona:", repr(e1))
        gdf = gpd.read_file(path, engine="fiona", **kwargs)
    if gdf.crs is None and crs_fallback is not None:
        gdf = gdf.set_crs(crs_fallback, allow_override=True)
    return gdf

class PiecewiseNorm(Normalize):
    def __init__(self, x, y, clip=False):
        super().__init__(vmin=min(x), vmax=max(x), clip=clip)
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
        if not (np.all(np.diff(self.x) > 0) and np.all(np.diff(self.y) > 0)):
            raise ValueError("x and y must be strictly increasing.")
    def __call__(self, value, clip=None):
        v = np.asarray(value, float)
        if (clip is True) or (self.clip is True):
            v = np.clip(v, self.vmin, self.vmax)
        return np.interp(v, self.x, self.y)
    def inverse(self, value):
        v = np.asarray(value, float)
        return np.interp(v, self.y, self.x)

def build_segment_cmap(y5=Y_5, y20=Y_20, y50=Y_50):
    stops = [
        (0.00, (0.90, 1.00, 0.90)),
        (y5,   (0.10, 0.65, 0.20)),

        (y5,   (0.88, 1.00, 1.00)),
        (y20,  (0.00, 0.70, 0.75)),

        (y20,  (0.90, 0.95, 1.00)),
        (y50,  (0.10, 0.30, 0.85)),

        (y50,  (1.00, 0.92, 0.92)),
        (1.00, (0.55, 0.00, 0.00)),
    ]
    return LinearSegmentedColormap.from_list("payback_piecewise", stops, N=256)

def robust_median_positive(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s[np.isfinite(s) & (s > 0)]
    return float(np.median(s)) if len(s) else np.nan

def benefit_hat(type_letter, land_price, area_m2, epop, maint_yen_per_m2_yr):
    p = BENEFIT_PARAMS[type_letter]
    return (p["c"]
            * np.power(land_price, p["a_lp"])
            * np.power(area_m2, p["a_area"])
            * np.power(np.maximum(epop, 1.0), p["a_epop"])
            * np.power(maint_yen_per_m2_yr, p["a_maint"]))

def payback_hat(land_price, area_m2, benefit_yen_yr, maint_yen_per_m2_yr):
    land_cost = land_price * area_m2
    annual_maint = maint_yen_per_m2_yr * area_m2
    net = benefit_yen_yr - annual_maint
    T = np.full_like(net, np.nan, dtype=float)
    ok = net > EPS
    T[ok] = land_cost[ok] / net[ok]
    return np.clip(T, 0.0, CAP_YEARS), ok

base = safe_read_shp(BASE_SHP, crs_fallback=DID_CRS_FALLBACK)
base_geom = base.unary_union

did = safe_read_shp(DID_SHP, columns_needed=[POP_COL], crs_fallback=DID_CRS_FALLBACK)
did = did.to_crs(base.crs)

did_clip = gpd.overlay(
    did,
    gpd.GeoDataFrame(geometry=[base_geom], crs=base.crs),
    how="intersection"
)
did_clip[POP_COL] = pd.to_numeric(did_clip[POP_COL], errors="coerce").fillna(0.0)
pop_pts = did_clip[did_clip[POP_COL] > 0].copy()
pop_pts["geometry"] = pop_pts.geometry.representative_point()

base_ll = gpd.GeoSeries([base_geom], crs=base.crs).to_crs("EPSG:4326").iloc[0]
cent = base_ll.centroid
lon, lat = cent.x, cent.y
utm_zone = int((lon + 180) // 6) + 1
utm_epsg = 32600 + utm_zone if lat >= 0 else 32700 + utm_zone
PROJ_CRS = f"EPSG:{utm_epsg}"

base_m = gpd.GeoSeries([base_geom], crs=base.crs).to_crs(PROJ_CRS).iloc[0]
boundary = gpd.GeoSeries([base_m], crs=PROJ_CRS).boundary

pop_pts_m = pop_pts.to_crs(PROJ_CRS)
pop_xy = np.vstack([pop_pts_m.geometry.x.values, pop_pts_m.geometry.y.values]).T
pop_w  = pop_pts_m[POP_COL].values.astype(float)

print(f"[pop points] n={len(pop_pts_m)} | pop sum(raw)={pop_w.sum():,.2f}")
print("[projection CRS]", PROJ_CRS)

TARGET_TOTAL_POP = 43_000_000
cur_pop = float(np.nansum(pop_w))
POP_SCALE = TARGET_TOTAL_POP / cur_pop if cur_pop > 0 else 1.0

print(f"[POP_SCALE] cur_pop={cur_pop:,.2f} | target={TARGET_TOTAL_POP:,.0f} | scale={POP_SCALE:.4f}")

pop_w = pop_w * POP_SCALE

print(f"[pop sum(scaled)] {float(np.nansum(pop_w)):,.2f}")

parks = pd.read_csv(PARKS_CSV, encoding="utf-8-sig")

type_col = "park_class_name" if "park_class_name" in parks.columns else None
if type_col is None:
    raise KeyError("Cannot find park_class_name in PARKS_CSV.")
parks[type_col] = parks[type_col].astype(str).str.strip().str.upper()

name_to_abcde = {
    "CITY BLOCK PARK": "A",
    "BLOCK PARK": "A",
    "NEIGHBORHOOD PARK": "B",
    "DISTRICT PARK": "C",
    "COMPREHENSIVE PARK": "D",
    "REGIONAL PARK": "E",
}
parks["abcde"] = parks[type_col].map(name_to_abcde)

lon_candidates = [c for c in ["Lng","lon","longitude","Lon","LONG"] if c in parks.columns]
lat_candidates = [c for c in ["Lat","lat","latitude","LAT"] if c in parks.columns]
if not lon_candidates or not lat_candidates:
    raise KeyError("PARKS_CSV missing lon/lat columns.")
LON_COL, LAT_COL = lon_candidates[0], lat_candidates[0]

area_candidates = [c for c in ["area","Area m2","Area_m2","area_m2","Area m2 "] if c in parks.columns]
maint_candidates = [c for c in ["unit_maintenance_yen_per_m2_2024","unit_maintenance_yen_per_m2","maint_yen_per_m2","maint"] if c in parks.columns]
if not area_candidates or not maint_candidates:
    raise KeyError("PARKS_CSV missing area or maint intensity column.")
AREA_COL = area_candidates[0]
MAINT_COL = maint_candidates[0]

for c in [LON_COL, LAT_COL, AREA_COL, MAINT_COL]:
    parks[c] = pd.to_numeric(parks[c], errors="coerce")
parks = parks[np.isfinite(parks[LON_COL]) & np.isfinite(parks[LAT_COL])].copy()

gparks = gpd.GeoDataFrame(
    parks,
    geometry=gpd.points_from_xy(parks[LON_COL], parks[LAT_COL]),
    crs="EPSG:4326"
).to_crs(PROJ_CRS)

MAINT_A = robust_median_positive(gparks.loc[gparks["abcde"]=="A", MAINT_COL])
MAINT_B = robust_median_positive(gparks.loc[gparks["abcde"]=="B", MAINT_COL])
print(f"[maint ref] A median={MAINT_A:.4g} | B median={MAINT_B:.4g}")
if not (np.isfinite(MAINT_A) and np.isfinite(MAINT_B)):
    raise ValueError("MAINT_A/MAINT_B is NaN. Check MAINT_COL values for A/B in PARKS_CSV.")

park_xy = np.vstack([gparks.geometry.x.values, gparks.geometry.y.values]).T
park_type = gparks["abcde"].values.astype(object)

sigma_p = np.array([SIGMA.get(t, np.nan) for t in park_type], dtype=float)
beta_p  = np.array([BETA.get(t, np.nan)  for t in park_type], dtype=float)
okp = np.isfinite(sigma_p) & np.isfinite(beta_p)

park_xy2 = park_xy[okp]
sigma_p2 = sigma_p[okp]
beta_p2  = beta_p[okp]

dx = pop_xy[:, 0:1] - park_xy2[None, :, 0]
dy = pop_xy[:, 1:2] - park_xy2[None, :, 1]
dist = np.sqrt(dx*dx + dy*dy)

mask = dist <= R_M
dist_safe = np.maximum(dist, 1.0)

W = sigma_p2[None, :] * np.power(dist_safe, beta_p2[None, :])
W[~mask] = 0.0
sum_w_existing = W.sum(axis=1)

print("[sum_w_existing] min/median/max:",
      float(np.min(sum_w_existing)), float(np.median(sum_w_existing)), float(np.max(sum_w_existing)))

minx, miny, maxx, maxy = base_m.bounds
xs = np.arange(minx, maxx + GRID_STEP_M, GRID_STEP_M)
ys = np.arange(miny, maxy + GRID_STEP_M, GRID_STEP_M)
XX, YY = np.meshgrid(xs, ys)
grid_xy = np.vstack([XX.ravel(), YY.ravel()]).T

grid_gdf = gpd.GeoDataFrame(geometry=[Point(xy) for xy in grid_xy], crs=PROJ_CRS)
inside = grid_gdf.within(base_m).values
grid_in = grid_xy[inside]
grid_gdf_in = grid_gdf.loc[inside].copy()

print(f"[grid] total={len(grid_xy):,} | inside={len(grid_in):,} | step={GRID_STEP_M}m")

lp = safe_read_shp(LANDPRICE_SHP, crs_fallback=LP_CRS_FALLBACK).to_crs(PROJ_CRS)
if LP_VALUE_COL not in lp.columns:
    raise KeyError(f"LP_VALUE_COL={LP_VALUE_COL} not found in LANDPRICE_SHP.")
lp[LP_VALUE_COL] = pd.to_numeric(lp[LP_VALUE_COL], errors="coerce")

joined = gpd.sjoin(grid_gdf_in, lp[[LP_VALUE_COL, "geometry"]], how="left", predicate="within")
lp_grid = joined[LP_VALUE_COL].to_numpy(float)

miss = ~np.isfinite(lp_grid)
if miss.any():
    nearest = gpd.sjoin_nearest(grid_gdf_in.loc[miss], lp[[LP_VALUE_COL, "geometry"]], how="left")
    lp_grid[miss] = nearest[LP_VALUE_COL].to_numpy(float)

lp_grid = np.where(np.isfinite(lp_grid) & (lp_grid > 0), lp_grid, np.nan)
valid_lp = np.isfinite(lp_grid)

print("[land price] finite share:", float(np.mean(valid_lp)))
print("[land price] median (finite):", float(np.nanmedian(lp_grid)))

def compute_epop(grid_in_xy, type_letter, chunk=6000):
    sig = SIGMA[type_letter]
    bet = BETA[type_letter]
    out = np.zeros(len(grid_in_xy), dtype=float)

    for s in range(0, len(grid_in_xy), chunk):
        e = min(len(grid_in_xy), s + chunk)
        g = grid_in_xy[s:e]

        dx = g[:, 0:1] - pop_xy[None, :, 0]
        dy = g[:, 1:2] - pop_xy[None, :, 1]
        d = np.sqrt(dx*dx + dy*dy)

        m = d <= R_M
        d_safe = np.maximum(d, 1.0)
        w_c = sig * np.power(d_safe, bet)
        w_c[~m] = 0.0

        # Candidate exposure is its population share after competition with all existing parks.
        denom = sum_w_existing[None, :] + w_c
        share = np.where(denom > 0, w_c / denom, 0.0)
        out[s:e] = (share * pop_w[None, :]).sum(axis=1)

    return out

Epop_A = compute_epop(grid_in, "A")
Epop_B = compute_epop(grid_in, "B")

TA = np.full(len(grid_in), np.nan, float)
TB = np.full(len(grid_in), np.nan, float)

benA = benefit_hat("A", lp_grid[valid_lp], A_AREA_M2, Epop_A[valid_lp], MAINT_A)
benB = benefit_hat("B", lp_grid[valid_lp], B_AREA_M2, Epop_B[valid_lp], MAINT_B)

TA_valid, okA = payback_hat(lp_grid[valid_lp], A_AREA_M2, benA, MAINT_A)
TB_valid, okB = payback_hat(lp_grid[valid_lp], B_AREA_M2, benB, MAINT_B)

TA[valid_lp] = TA_valid
TB[valid_lp] = TB_valid

TA_full = np.full(len(grid_xy), np.nan, float)
TB_full = np.full(len(grid_xy), np.nan, float)
TA_full[inside] = TA
TB_full[inside] = TB

TA_img = TA_full.reshape(YY.shape)
TB_img = TB_full.reshape(YY.shape)

NORM = PiecewiseNorm(
    x=[0, 5, 20, 50, 300],
    y=[0.00, Y_5, Y_20, Y_50, 1.00],
    clip=True
)
CMAP = build_segment_cmap(Y_5, Y_20, Y_50).copy()
CMAP.set_bad("white")

area_vals = pd.to_numeric(gparks[AREA_COL], errors="coerce").to_numpy(float)
area_vals = np.where(np.isfinite(area_vals) & (area_vals > 0), area_vals, np.nan)

sqrtA = np.sqrt(area_vals)
s_raw = (sqrtA / 25.0)**2
s = np.clip(s_raw * 100.0 * MARKER_EXTRA_SHRINK, S_MIN, S_MAX)

def draw_map(ax, img, title):
    im = ax.imshow(
        img,
        origin="lower",
        extent=[minx, maxx, miny, maxy],
        cmap=CMAP,
        norm=NORM,
        interpolation="bilinear"
    )

    boundary.plot(ax=ax, linewidth=1.1, color="black")

    ax.scatter(
        gparks.geometry.x.values,
        gparks.geometry.y.values,
        s=s,
        facecolors="none",
        edgecolors="black",
        linewidths=0.30,
        alpha=0.30,
        zorder=3
    )

    ax.set_title(title, pad=10 * FONT_SCALE)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_axis_off()

    return im

fig, axs = plt.subplots(1, 2, figsize=(18.0, 7.2), constrained_layout=True)

im0 = draw_map(axs[0], TA_img, f"Type A ({A_HA:.2f}ha)")
im1 = draw_map(axs[1], TB_img, f"Type B ({B_HA:.2f}ha)")

cb = fig.colorbar(im0, ax=axs, fraction=0.046, pad=0.02)
cb.set_label("Predicted payback time (years)", fontsize=10 * FONT_SCALE)
cb.set_ticks([0, 5, 20, 50, 100, 200, 300])
cb.ax.tick_params(labelsize=9 * FONT_SCALE)

plt.show()

def diagnose(arr, name):
    a = arr.copy()
    finite = np.isfinite(a)
    print(f"\n[{name}] finite share (inside grid) = {finite.mean()*100:.2f}% | NaN(white)={100 - finite.mean()*100:.2f}%")

    lp_missing = (~valid_lp).mean()*100
    print(f"  land-price missing (before fill+clean) within grid_in ≈ {lp_missing:.2f}%")

    return

diagnose(TA, "Type A")
diagnose(TB, "Type B")

print("\n[feasibility among land-price-valid points]")
print(f"  Type A: infeasible share ≈ {(1.0 - okA.mean())*100:.2f}%")
print(f"  Type B: infeasible share ≈ {(1.0 - okB.mean())*100:.2f}%")


In [ ]:
# Summarize candidate-grid area shares by payback category.
def area_share_report(T_vec, grid_step_m, name):
    """
    T_vec: length = len(grid_in) payback values for inside-grid points (NaN allowed)
    """
    T = np.asarray(T_vec, float)
    cell_area = (grid_step_m ** 2)

    total_cells = len(T)
    total_area = total_cells * cell_area

    finite = np.isfinite(T)
    nan_cells = np.sum(~finite)

    b0 = np.sum(finite & (T >= 0)  & (T < 5))
    b1 = np.sum(finite & (T >= 5)  & (T < 20))
    b2 = np.sum(finite & (T >= 20) & (T <= 50))
    b3 = np.sum(finite & (T > 50))

    def pct(n):
        return 100.0 * (n * cell_area) / total_area

    print(f"\n[{name}] Area shares (grid step={grid_step_m} m; cell={cell_area:,.0f} m²)")
    print(f"  0–5 years     : {pct(b0):6.2f}%")
    print(f"  5–20 years    : {pct(b1):6.2f}%")
    print(f"  20–50 years   : {pct(b2):6.2f}%")
    print(f"  >50 years     : {pct(b3):6.2f}%")
    print(f"  NaN (white)   : {pct(nan_cells):6.2f}%")
    print(f"  (check sum)   : {pct(b0+b1+b2+b3+nan_cells):6.2f}%")

area_share_report(TA, GRID_STEP_M, f"Type A ({A_HA:.2f} ha)")
area_share_report(TB, GRID_STEP_M, f"Type B ({B_HA:.2f} ha)")


In [ ]:
# Calculate local elasticities and one-at-a-time sensitivity curves.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PARAMS = {
    "A": {"Ct": 0.0313952, "b1": 0.8123, "b2": 0.9647, "b3": 0.1073, "b4": 0.2503},
    "B": {"Ct": 0.3566980, "b1": 0.7719, "b2": 0.7120, "b3": 0.0990, "b4": 0.2885},
}

BASE = {
    "land_price": 4.0,
    "kcon": 1.0,
    "area": 1.0,
    "Epop": 1.0,
    "maint": 1.0,
}

def B_t(t, land_price, area, Epop, maint):
    p = PARAMS[t]
    return (
        p["Ct"]
        * (land_price ** p["b1"])
        * (area ** p["b2"])
        * (Epop ** p["b3"])
        * (maint ** p["b4"])
    )

def C0(area, land_price, kcon):
    return area * (land_price + kcon)

def T_payback(t, land_price, area, Epop, maint, kcon):
    return C0(area, land_price, kcon) / B_t(t, land_price, area, Epop, maint)

def elasticities_at_baseline():
    lp0 = BASE["land_price"]
    kc0 = BASE["kcon"]
    s_land = lp0 / (lp0 + kc0)
    s_kcon = kc0 / (lp0 + kc0)

    rows = []
    for t in ["A", "B"]:
        p = PARAMS[t]
        rows.append([t, "Epop",       -p["b3"]])
        rows.append([t, "maint",      -p["b4"]])
        rows.append([t, "area",       1.0 - p["b2"]])
        rows.append([t, "land_price", s_land - p["b1"]])
        rows.append([t, "kcon",       s_kcon])

    elast = pd.DataFrame(rows, columns=["Type", "Variable", "elasticity_dlnT_dlnX"])
    return elast, s_land, s_kcon

def run_sensitivity(dmin=0.0, dmax=0.20, n=81):
    if dmin < 0:
        raise ValueError("This version is for dmin >= 0 (0% to +20%).")
    deltas = np.linspace(dmin, dmax, n)
    variables = ["land_price", "area", "Epop", "maint", "kcon"]

    lp0, a0, e0, m0, kc0 = (BASE["land_price"], BASE["area"], BASE["Epop"], BASE["maint"], BASE["kcon"])
    if min(lp0, a0, e0, m0, kc0) <= 0:
        raise ValueError("All BASE values must be > 0.")

    elast, _, _ = elasticities_at_baseline()
    elast_map = {(r["Type"], r["Variable"]): r["elasticity_dlnT_dlnX"] for _, r in elast.iterrows()}

    records = []
    for t in ["A", "B"]:
        T0 = T_payback(t, lp0, a0, e0, m0, kc0)

        for var in variables:
            e_elast = elast_map[(t, var)]
            for d in deltas:

                lp = lp0 * (1.0 + d) if var == "land_price" else lp0
                a  = a0  * (1.0 + d) if var == "area"       else a0
                ep = e0  * (1.0 + d) if var == "Epop"       else e0
                ma = m0  * (1.0 + d) if var == "maint"      else m0
                kc = kc0 * (1.0 + d) if var == "kcon"       else kc0

                T1 = T_payback(t, lp, a, ep, ma, kc)
                pct_exact = (T1 / T0 - 1.0) * 100.0

                pct_elastic = (np.exp(e_elast * np.log(1.0 + d)) - 1.0) * 100.0

                records.append({
                    "Type": t,
                    "Variable": var,
                    "delta": d,
                    "delta_pct": d * 100.0,
                    "pct_change_T_exact": pct_exact,
                    "pct_change_T_elastic": pct_elastic,
                })

    return pd.DataFrame(records), elast

def plot_sensitivity(df, out_png="payback_sensitivity_0_20_solid.png"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    order = ["land_price", "area", "Epop", "maint", "kcon"]

    for ax, t in zip(axes, ["A", "B"]):
        sub = df[df["Type"] == t]
        for var in order:
            tmp = sub[sub["Variable"] == var].sort_values("delta")
            ax.plot(tmp["delta_pct"], tmp["pct_change_T_exact"], label=var)

        ax.axhline(0, linewidth=1)
        ax.set_title(f"Type {t}")
        ax.set_xlabel("Input change (%)")

    axes[0].set_ylabel("Payback time change (%)")
    axes[1].legend(frameon=False, fontsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

if __name__ == "__main__":
    elast, s_land, s_kcon = elasticities_at_baseline()
    print("=== Baseline shares in C0 = area*(land_price + kcon) ===")
    print(f"s_land = land_price/(land_price+kcon) = {s_land:.6f}")
    print(f"s_kcon = kcon/(land_price+kcon)       = {s_kcon:.6f}\n")

    print("=== Elasticities: d ln T / d ln X (baseline) ===")
    print(elast.to_string(index=False))
    print()

    df, _ = run_sensitivity(dmin=0.0, dmax=0.20, n=81)

    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    elast.to_csv(output_dir / "payback_elasticities.csv", index=False)
    df.to_csv(output_dir / "payback_sensitivity_curves_0_20.csv", index=False)

    print("=== Sensitivity dataframe (head) ===")
    print(df.head(10).to_string(index=False))
    print("\nSaved files:")
    print(" - payback_elasticities.csv")
    print(" - payback_sensitivity_curves_0_20.csv")

    plot_sensitivity(df, out_png=output_dir / "payback_sensitivity_0_20_solid.png")
    print(" - payback_sensitivity_0_20_solid.png")


In [ ]:
# Fit and evaluate the planning-stage benefit and payback classification models.
"""
Binary-main framework for planning-stage benefit prediction
and 2% discounted payback translation.

Main validation target
----------------------
within_50y vs not_within_50y

Auxiliary outputs retained
--------------------------
- predicted payback year
- predicted 3-class payback state:
    within_50y / beyond_50y / never
- maps and diagnostics

Design principles
-----------------
1) Predict annual benefit first
2) Translate predicted benefit into discounted payback using the same 2% accounting rule
3) Main model validation focuses on binary recoverability within 50 years
4) Keep methods conservative and auditable
"""

import os
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

INPUT_CSV = r"outputs/ab_modeling_table_r2.csv"
OUT_DIR = os.path.dirname(INPUT_CSV)

OUT_CSV = os.path.join(OUT_DIR, "ab_modeling_with_predictions_r2_binary_main.csv")
OUT_METRICS = os.path.join(OUT_DIR, "metrics_summary_r2_binary_main.txt")

FIG_BENEFIT = os.path.join(OUT_DIR, "fig_benefit_obs_vs_pred_r2_binary_main.png")
FIG_CM_BIN_BASE = os.path.join(OUT_DIR, "fig_confusion_matrix_binary_baseline_r2.png")
FIG_CM_BIN_UP = os.path.join(OUT_DIR, "fig_confusion_matrix_binary_upgraded_r2.png")
FIG_MAP_BIN = os.path.join(OUT_DIR, "fig_binary_within50y_map_upgraded_r2.png")
FIG_MAP_3CLASS = os.path.join(OUT_DIR, "fig_3class_map_upgraded_r2_binary_main.png")
FIG_PAYBACK = os.path.join(OUT_DIR, "fig_payback_year_obs_vs_pred_finite_r2_binary_main.png")

R = 0.02
EVAL_YEARS = 50
# Five-fold cross-validation evaluates predictions on held-out parks and limits in-sample optimism.
N_SPLITS = 5
RANDOM_STATE = 42
TARGET = "annual_benefit_yen_2019"

MAIN_RATE_TAG = "r2"

CLASS_ORDER_3 = ["within_50y", "beyond_50y", "never"]

HGBR_PARAMS = dict(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=250,
    max_depth=3,
    min_samples_leaf=40,
    l2_regularization=1.0,
    random_state=RANDOM_STATE,
)

def safe_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def classify_payback_3(y):
    if pd.isna(y):
        return np.nan
    if np.isinf(y):
        return "never"
    if y <= 50:
        return "within_50y"
    return "beyond_50y"

def classify_within50y_binary(y):
    if pd.isna(y):
        return np.nan
    if np.isinf(y):
        return 0
    return int(y <= 50)

def annuity_factor(n=50, r=0.02):
    return (1 - (1 + r) ** (-n)) / r

def payback_year_from_net_benefit(net_benefit, upfront_cost, r=0.02):
    if pd.isna(net_benefit) or pd.isna(upfront_cost):
        return np.nan
    if upfront_cost <= 0:
        return 0.0
    if net_benefit <= 0:
        return np.inf
    if (net_benefit / r) <= upfront_cost:
        return np.inf

    inner = 1 - r * upfront_cost / net_benefit
    if inner <= 0:
        return np.inf

    t = -np.log(inner) / np.log(1 + r)
    if not np.isfinite(t):
        return np.inf
    return float(np.ceil(t))

def draw_confusion_matrix(cm, labels, out_png, title):
    fig, ax = plt.subplots(figsize=(5.8, 5))
    ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=20)
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    ax.set_ylabel("Observed")
    ax.set_xlabel("Predicted")
    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def print_and_write(lines, filepath):
    text = "\n".join(lines)
    print(text)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

def add_features(df):
    out = df.copy()

    out["E_pop_per_m2"] = out["E_pop"] / out["area_m2"].clip(lower=1.0)
    out["investment_per_m2"] = out["one_time_investment_2019"] / out["area_m2"].clip(lower=1.0)

    out["lp_land_price"] = np.log1p(out["land_price"].clip(lower=0))
    out["lp_area_m2"] = np.log1p(out["area_m2"].clip(lower=0))
    out["lp_E_pop"] = np.log1p(out["E_pop"].clip(lower=0))
    out["lp_maint"] = np.log1p(out["maint"].clip(lower=0))
    out["lp_E_pop_per_m2"] = np.log1p(out["E_pop_per_m2"].clip(lower=0))
    out["lp_investment_per_m2"] = np.log1p(out["investment_per_m2"].clip(lower=0))

    return out

def translate_payback(df, pred_benefit_col, prefix):
    out = df.copy()

    out[f"{prefix}_annual_net_benefit_2019"] = out[pred_benefit_col] - out["annual_om_cost_2019"]

    out[f"{prefix}_payback_year_{MAIN_RATE_TAG}"] = out.apply(
        lambda row: payback_year_from_net_benefit(
            row[f"{prefix}_annual_net_benefit_2019"],
            row["one_time_investment_2019"],
            r=R,
        ),
        axis=1
    )

    out[f"{prefix}_payback_class_3"] = out[f"{prefix}_payback_year_{MAIN_RATE_TAG}"].apply(classify_payback_3)
    out[f"{prefix}_within50y_pred"] = out[f"{prefix}_payback_year_{MAIN_RATE_TAG}"].apply(classify_within50y_binary)

    AF50 = annuity_factor(EVAL_YEARS, R)
    out[f"{prefix}_recoverability_score_50y"] = (
        out[f"{prefix}_annual_net_benefit_2019"] * AF50 / out["one_time_investment_2019"]
    )

    return out

def oof_model_by_type(df, feature_cols, model_kind, pred_prefix):
    out = df.copy()
    out[f"{pred_prefix}_pred_log_benefit"] = np.nan
    out[f"{pred_prefix}_pred_benefit"] = np.nan

    metrics_rows = []

    for ptype in ["A", "B"]:
        sub = out[out["park_class"] == ptype].copy()
        idx = sub.index

        X = sub[feature_cols].copy()
        y_log = np.log1p(sub[TARGET].clip(lower=0))
        oof_pred_log = np.full(len(sub), np.nan)

        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

        for tr, te in kf.split(X):
            X_tr = X.iloc[tr].copy()
            X_te = X.iloc[te].copy()
            y_tr = y_log.iloc[tr].copy()

            if model_kind == "ols":
                model = LinearRegression()
            elif model_kind == "hgbr":
                model = HistGradientBoostingRegressor(**HGBR_PARAMS)
            else:
                raise ValueError("model_kind must be 'ols' or 'hgbr'")

            model.fit(X_tr, y_tr)
            pred_te_log = model.predict(X_te)

            lo = np.quantile(y_tr, 0.01)
            hi = np.quantile(y_tr, 0.99)
            pred_te_log = np.clip(pred_te_log, lo, hi)

            oof_pred_log[te] = pred_te_log

        sub[f"{pred_prefix}_pred_log_benefit"] = oof_pred_log
        sub[f"{pred_prefix}_pred_benefit"] = np.clip(np.expm1(oof_pred_log), 0, None)

        out.loc[idx, f"{pred_prefix}_pred_log_benefit"] = sub[f"{pred_prefix}_pred_log_benefit"].values
        out.loc[idx, f"{pred_prefix}_pred_benefit"] = sub[f"{pred_prefix}_pred_benefit"].values

        y_true = sub[TARGET].values
        y_true_log = np.log1p(np.clip(y_true, 0, None))
        y_pred = sub[f"{pred_prefix}_pred_benefit"].values
        y_pred_log = sub[f"{pred_prefix}_pred_log_benefit"].values

        metrics_rows.append({
            "park_class": ptype,
            "model": model_kind,
            "n": int(len(sub)),
            "r2_log": float(r2_score(y_true_log, y_pred_log)),
            "mae_log": float(mean_absolute_error(y_true_log, y_pred_log)),
            "rmse_log": float(rmse(y_true_log, y_pred_log)),
            "r2_level": float(r2_score(y_true, y_pred)),
            "mae_level": float(mean_absolute_error(y_true, y_pred)),
            "rmse_level": float(rmse(y_true, y_pred)),
        })

    return out, pd.DataFrame(metrics_rows)

def evaluate_binary(df, pred_bin_col):
    mask = df["obs_within50y_binary"].notna() & df[pred_bin_col].notna()
    y_true = df.loc[mask, "obs_within50y_binary"].astype(int)
    y_pred = df.loc[mask, pred_bin_col].astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    f1_pos = f1_score(y_true, y_pred, pos_label=1)
    precision_pos = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_pos = recall_score(y_true, y_pred, pos_label=1, zero_division=0)

    report = classification_report(
        y_true, y_pred,
        labels=[1, 0],
        target_names=["within_50y", "not_within_50y"],
        digits=4,
        zero_division=0
    )

    return {
        "cm": cm,
        "acc": acc,
        "macro_f1": macro_f1,
        "f1_within50y": f1_pos,
        "precision_within50y": precision_pos,
        "recall_within50y": recall_pos,
        "report": report,
    }

def evaluate_three_class(df, pred_class_col):
    mask = df["obs_payback_class_3"].notna() & df[pred_class_col].notna()
    y_true = df.loc[mask, "obs_payback_class_3"]
    y_pred = df.loc[mask, pred_class_col]

    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER_3)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=CLASS_ORDER_3, average="macro")
    report = classification_report(
        y_true, y_pred,
        labels=CLASS_ORDER_3,
        digits=4,
        zero_division=0
    )

    return {
        "cm": cm,
        "acc": acc,
        "macro_f1": macro_f1,
        "report": report,
    }

def evaluate_finite_payback_error(df, pred_year_col):
    finite_mask = np.isfinite(df[f"discounted_payback_years_{MAIN_RATE_TAG}"]) & np.isfinite(df[pred_year_col])

    if finite_mask.sum() > 0:
        obs = df.loc[finite_mask, f"discounted_payback_years_{MAIN_RATE_TAG}"].values
        pred = df.loc[finite_mask, pred_year_col].values
        mae = mean_absolute_error(obs, pred)
        rmse_v = rmse(obs, pred)
        med_ae = np.median(np.abs(obs - pred))
    else:
        mae = np.nan
        rmse_v = np.nan
        med_ae = np.nan

    return {
        "n": int(finite_mask.sum()),
        "mae": mae,
        "rmse": rmse_v,
        "median_ae": med_ae,
    }

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

required_cols = [
    "osm_id_norm", "park_class", "Lng", "Lat",
    "land_price", "area_m2", "E_pop", "maint",
    "annual_om_cost_2019", "one_time_investment_2019",
    TARGET, f"discounted_payback_years_{MAIN_RATE_TAG}"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

num_cols = [
    "Lng", "Lat", "land_price", "area_m2", "E_pop", "maint",
    "annual_om_cost_2019", "one_time_investment_2019",
    TARGET, f"discounted_payback_years_{MAIN_RATE_TAG}"
]
df = safe_numeric(df, num_cols)

df["park_class"] = df["park_class"].astype(str).str.strip()
df = df[df["park_class"].isin(["A", "B"])].copy()

df["obs_payback_class_3"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"].apply(classify_payback_3)
df["obs_within50y_binary"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"].apply(classify_within50y_binary)

df = add_features(df)

needed = [
    "lp_land_price", "lp_area_m2", "lp_E_pop", "lp_maint",
    "lp_E_pop_per_m2", "lp_investment_per_m2",
    "annual_om_cost_2019", "one_time_investment_2019",
    TARGET
]
valid_mask = df[needed].notna().all(axis=1)
for c in ["annual_om_cost_2019", "one_time_investment_2019", TARGET]:
    valid_mask &= (df[c] >= 0)

work = df[valid_mask].copy().reset_index(drop=True)

print("Total A/B rows:", len(df))
print("Rows used for modeling:", len(work))
print("\nObserved 3-class distribution:")
print(work["obs_payback_class_3"].value_counts(dropna=False))
print("\nObserved binary distribution:")
print(work["obs_within50y_binary"].value_counts(dropna=False).rename({1: "within_50y", 0: "not_within_50y"}))

BASELINE_FEATURES = [
    "lp_land_price",
    "lp_area_m2",
    "lp_E_pop",
    "lp_maint",
]

UPGRADED_FEATURES = [
    "lp_land_price",
    "lp_area_m2",
    "lp_E_pop",
    "lp_maint",
    "lp_E_pop_per_m2",
    "lp_investment_per_m2",
]

base_df, base_metrics = oof_model_by_type(
    work,
    feature_cols=BASELINE_FEATURES,
    model_kind="ols",
    pred_prefix="baseline",
)
base_df = translate_payback(base_df, "baseline_pred_benefit", "baseline")

up_df, up_metrics = oof_model_by_type(
    work,
    feature_cols=UPGRADED_FEATURES,
    model_kind="hgbr",
    pred_prefix="upgraded",
)
up_df = translate_payback(up_df, "upgraded_pred_benefit", "upgraded")

final_df = up_df.copy()
for c in [
    "baseline_pred_log_benefit", "baseline_pred_benefit",
    "baseline_annual_net_benefit_2019",
    f"baseline_payback_year_{MAIN_RATE_TAG}",
    "baseline_payback_class_3",
    "baseline_within50y_pred",
    "baseline_recoverability_score_50y"
]:
    final_df[c] = base_df[c].values

base_bin = evaluate_binary(final_df, "baseline_within50y_pred")
up_bin = evaluate_binary(final_df, "upgraded_within50y_pred")

base_3 = evaluate_three_class(final_df, "baseline_payback_class_3")
up_3 = evaluate_three_class(final_df, "upgraded_payback_class_3")

base_finite = evaluate_finite_payback_error(final_df, f"baseline_payback_year_{MAIN_RATE_TAG}")
up_finite = evaluate_finite_payback_error(final_df, f"upgraded_payback_year_{MAIN_RATE_TAG}")

y_true_bin = final_df["obs_within50y_binary"].dropna().astype(int)
dummy_bin_acc = (y_true_bin == 0).mean()

fig, ax = plt.subplots(figsize=(7, 6))
for ptype, marker in zip(["A", "B"], ["o", "^"]):
    sub = final_df[final_df["park_class"] == ptype]
    ax.scatter(
        sub[TARGET], sub["upgraded_pred_benefit"],
        s=18, alpha=0.5, marker=marker, label=f"Type {ptype}"
    )
xy_max = max(final_df[TARGET].max(), final_df["upgraded_pred_benefit"].max())
ax.plot([0, xy_max], [0, xy_max], linestyle="--", linewidth=1)
ax.set_xlabel("Observed annual benefit (JPY, 2019)")
ax.set_ylabel("Predicted annual benefit (JPY, 2019)")
ax.set_title("Observed vs predicted annual benefit (binary-main upgraded model, r=2%)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_BENEFIT, dpi=300, bbox_inches="tight")
plt.close(fig)

draw_confusion_matrix(
    base_bin["cm"],
    labels=["within_50y", "not_within_50y"],
    out_png=FIG_CM_BIN_BASE,
    title="Binary confusion matrix (baseline log-log OLS, r=2%)"
)
draw_confusion_matrix(
    up_bin["cm"],
    labels=["within_50y", "not_within_50y"],
    out_png=FIG_CM_BIN_UP,
    title="Binary confusion matrix (upgraded conservative HGBR, r=2%)"
)

fig, ax = plt.subplots(figsize=(8, 8))
color_map_bin = {1: "#1b7837", 0: "#bdbdbd"}
label_map_bin = {1: "within_50y", 0: "not_within_50y"}
for cls in [1, 0]:
    sub = final_df[final_df["upgraded_within50y_pred"] == cls]
    ax.scatter(
        sub["Lng"], sub["Lat"],
        s=10, alpha=0.75,
        c=color_map_bin[cls],
        label=label_map_bin[cls]
    )
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Predicted recoverability within 50 years (upgraded model, r=2%)")
ax.legend(title="Predicted class")
fig.tight_layout()
fig.savefig(FIG_MAP_BIN, dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 8))
color_map_3 = {
    "within_50y": "#1b7837",
    "beyond_50y": "#91cf60",
    "never": "#bdbdbd",
}
for cls in CLASS_ORDER_3:
    sub = final_df[final_df["upgraded_payback_class_3"] == cls]
    ax.scatter(
        sub["Lng"], sub["Lat"],
        s=10, alpha=0.75,
        c=color_map_3[cls],
        label=cls
    )
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Predicted discounted payback classes (auxiliary 3-class map, r=2%)")
ax.legend(title="Predicted class")
fig.tight_layout()
fig.savefig(FIG_MAP_3CLASS, dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
mask_finite_both = (
    np.isfinite(final_df[f"discounted_payback_years_{MAIN_RATE_TAG}"]) &
    np.isfinite(final_df[f"upgraded_payback_year_{MAIN_RATE_TAG}"])
)
if mask_finite_both.sum() > 0:
    sub = final_df.loc[mask_finite_both].copy()
    for ptype, marker in zip(["A", "B"], ["o", "^"]):
        tmp = sub[sub["park_class"] == ptype]
        ax.scatter(
            tmp[f"discounted_payback_years_{MAIN_RATE_TAG}"],
            tmp[f"upgraded_payback_year_{MAIN_RATE_TAG}"],
            s=18, alpha=0.55, marker=marker, label=f"Type {ptype}"
        )
    pay_max = max(sub[f"discounted_payback_years_{MAIN_RATE_TAG}"].max(), sub[f"upgraded_payback_year_{MAIN_RATE_TAG}"].max())
    ax.plot([0, pay_max], [0, pay_max], linestyle="--", linewidth=1)
    ax.set_xlabel("Observed discounted payback year (finite only)")
    ax.set_ylabel("Predicted discounted payback year (finite only)")
    ax.set_title("Observed vs predicted payback year (finite-only, upgraded, r=2%)")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No finite-overlap cases", ha="center", va="center")
fig.tight_layout()
fig.savefig(FIG_PAYBACK, dpi=300, bbox_inches="tight")
plt.close(fig)

final_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

lines = []
lines.append("=== SAMPLE ===")
lines.append(f"Input CSV: {INPUT_CSV}")
lines.append(f"Total A/B rows: {len(df)}")
lines.append(f"Rows used for modeling: {len(work)}")
lines.append("")
lines.append("Observed 3-class distribution:")
lines.append(work["obs_payback_class_3"].value_counts(dropna=False).to_string())
lines.append("")
lines.append("Observed binary distribution:")
lines.append(work["obs_within50y_binary"].value_counts(dropna=False).rename({1: "within_50y", 0: "not_within_50y"}).to_string())
lines.append("")

lines.append("=== BENEFIT MODEL METRICS ===")
lines.append("Baseline (true log-log OLS):")
lines.append(base_metrics.to_string(index=False))
lines.append("")
lines.append("Upgraded (conservative HGBR):")
lines.append(up_metrics.to_string(index=False))
lines.append("")

lines.append("=== MAIN VALIDATION: BINARY WITHIN_50Y vs NOT_WITHIN_50Y ===")
lines.append("Baseline:")
lines.append(f"Accuracy = {base_bin['acc']:.4f}")
lines.append(f"Macro-F1 = {base_bin['macro_f1']:.4f}")
lines.append(f"F1 (within_50y) = {base_bin['f1_within50y']:.4f}")
lines.append(f"Precision (within_50y) = {base_bin['precision_within50y']:.4f}")
lines.append(f"Recall (within_50y) = {base_bin['recall_within50y']:.4f}")
lines.append("Classification report:")
lines.append(base_bin["report"])
lines.append("Confusion matrix (rows=observed, cols=predicted):")
lines.append(pd.DataFrame(
    base_bin["cm"],
    index=["within_50y", "not_within_50y"],
    columns=["within_50y", "not_within_50y"]
).to_string())
lines.append("")

lines.append("Upgraded:")
lines.append(f"Accuracy = {up_bin['acc']:.4f}")
lines.append(f"Macro-F1 = {up_bin['macro_f1']:.4f}")
lines.append(f"F1 (within_50y) = {up_bin['f1_within50y']:.4f}")
lines.append(f"Precision (within_50y) = {up_bin['precision_within50y']:.4f}")
lines.append(f"Recall (within_50y) = {up_bin['recall_within50y']:.4f}")
lines.append("Classification report:")
lines.append(up_bin["report"])
lines.append("Confusion matrix (rows=observed, cols=predicted):")
lines.append(pd.DataFrame(
    up_bin["cm"],
    index=["within_50y", "not_within_50y"],
    columns=["within_50y", "not_within_50y"]
).to_string())
lines.append("")
lines.append(f"Dummy baseline (always predict not_within_50y) accuracy = {dummy_bin_acc:.4f}")
lines.append("")

lines.append("=== AUXILIARY 3-CLASS TRANSLATION PERFORMANCE ===")
lines.append("Baseline:")
lines.append(f"Accuracy = {base_3['acc']:.4f}")
lines.append(f"Macro-F1 = {base_3['macro_f1']:.4f}")
lines.append("Classification report:")
lines.append(base_3["report"])
lines.append("")
lines.append("Upgraded:")
lines.append(f"Accuracy = {up_3['acc']:.4f}")
lines.append(f"Macro-F1 = {up_3['macro_f1']:.4f}")
lines.append("Classification report:")
lines.append(up_3["report"])
lines.append("")

lines.append("=== FINITE-ONLY PAYBACK ERROR ===")
lines.append("Baseline:")
lines.append(pd.Series(base_finite).to_string())
lines.append("")
lines.append("Upgraded:")
lines.append(pd.Series(up_finite).to_string())
lines.append("")

lines.append("=== BINARY CLASS SHARE BY TYPE (UPGRADED) ===")
for ptype in ["A", "B"]:
    lines.append(f"\nType {ptype}:")
    sub = final_df[final_df["park_class"] == ptype]
    obs_share = sub["obs_within50y_binary"].value_counts(normalize=True).reindex([1, 0]).fillna(0)
    pred_share = sub["upgraded_within50y_pred"].value_counts(normalize=True).reindex([1, 0]).fillna(0)
    tmp = pd.DataFrame({
        "observed_share": obs_share.values,
        "predicted_share": pred_share.values,
    }, index=["within_50y", "not_within_50y"])
    lines.append(tmp.to_string())

lines.append("")
lines.append("=== AUXILIARY 3-CLASS SHARE BY TYPE (UPGRADED) ===")
for ptype in ["A", "B"]:
    lines.append(f"\nType {ptype}:")
    sub = final_df[final_df["park_class"] == ptype]
    obs_share = sub["obs_payback_class_3"].value_counts(normalize=True).reindex(CLASS_ORDER_3).fillna(0)
    pred_share = sub["upgraded_payback_class_3"].value_counts(normalize=True).reindex(CLASS_ORDER_3).fillna(0)
    tmp = pd.DataFrame({
        "observed_share": obs_share,
        "predicted_share": pred_share,
    })
    lines.append(tmp.to_string())

lines.append("")
lines.append("=== OUTPUT FILES ===")
lines.append(OUT_CSV)
lines.append(OUT_METRICS)
lines.append(FIG_BENEFIT)
lines.append(FIG_CM_BIN_BASE)
lines.append(FIG_CM_BIN_UP)
lines.append(FIG_MAP_BIN)
lines.append(FIG_MAP_3CLASS)
lines.append(FIG_PAYBACK)

print_and_write(lines, OUT_METRICS)

print("\nSaved binary-main prediction table to:")
print(OUT_CSV)


In [ ]:
# Inspect the planning-stage prediction table.
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

csv = "outputs/ab_modeling_with_predictions_r2_binary_main.csv"
df = pd.read_csv(csv, encoding="utf-8-sig")

print("=== BENEFIT MODEL QUICK CHECK ===")
for c in ["park_class", "annual_benefit_yen_2019", "upgraded_pred_benefit", "baseline_pred_benefit"]:
    if c not in df.columns:
        print("missing:", c)

for typ in ["A", "B"]:
    sub = df[df["park_class"] == typ].copy()
    y = np.log1p(pd.to_numeric(sub["annual_benefit_yen_2019"], errors="coerce").clip(lower=0))
    yhat = np.log1p(pd.to_numeric(sub["upgraded_pred_benefit"], errors="coerce").clip(lower=0))
    mask = np.isfinite(y) & np.isfinite(yhat)
    if mask.sum():
        corr = np.corrcoef(y[mask], yhat[mask])[0,1]
        print(f"Type {typ} upgraded log-corr: {corr:.4f}, n={mask.sum()}")

print("\n=== BINARY MAIN CHECK ===")
y_true = pd.to_numeric(df["obs_within50y_binary"], errors="coerce")
y_pred = pd.to_numeric(df["upgraded_within50y_pred"], errors="coerce")
mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true = y_true[mask].astype(int)
y_pred = y_pred[mask].astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro-F1:", f1_score(y_true, y_pred, average="macro"))
print("F1 within_50y:", f1_score(y_true, y_pred, pos_label=1))
print("Dummy baseline acc:", (y_true == 0).mean())
print("\nConfusion matrix [rows true 1/0, cols pred 1/0]:")
print(confusion_matrix(y_true, y_pred, labels=[1,0]))
print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=[1,0], target_names=["within_50y","not_within_50y"], digits=4, zero_division=0))

print("\n=== BINARY CLASS SHARE BY TYPE (UPGRADED) ===")
for typ in ["A", "B"]:
    sub = df[df["park_class"] == typ].copy()
    obs = sub["obs_within50y_binary"].value_counts(normalize=True).reindex([1,0]).fillna(0)
    pred = sub["upgraded_within50y_pred"].value_counts(normalize=True).reindex([1,0]).fillna(0)
    print(f"\nType {typ}")
    print(pd.DataFrame({"observed_share": obs.values, "predicted_share": pred.values}, index=["within_50y","not_within_50y"]))


In [ ]:
# Plot observed and predicted annual benefits by park type.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "ab_modeling_with_predictions_r2_binary_main.csv")

EPS = 1e-12

TITLE_SCALE = 0.82
YTICK_SCALE = 0.82
FONT_SCALE = 1.00
FIGSIZE = (4.6, 3.4)

BOTTOM_FOR_LEGEND = 0.28
RIGHT_FOR_CBAR = 0.04

SAVE_FIG = False
OUT_FIG = os.path.join(OUT_DIR, "fig_R3_AB_benefit_only_r2.png")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11 * FONT_SCALE,
    "axes.titlesize": 12 * FONT_SCALE * TITLE_SCALE,
    "ytick.labelsize": 11 * FONT_SCALE * YTICK_SCALE,
    "xtick.labelsize": 11 * FONT_SCALE * 0.95,
    "axes.labelsize": 11 * FONT_SCALE,
    "legend.fontsize": 10 * FONT_SCALE,
    "font.family": "Times New Roman",
    "axes.unicode_minus": False,
})

BASE_CMAP_NAME = "viridis"
WHITEN = 0.58
DENSITY_ALPHA = 0.95

def make_pastel_cmap(name="viridis", whiten=0.58):
    cmap = plt.get_cmap(name)
    colors = cmap(np.linspace(0, 1, 256))
    colors[:, :3] = (1 - whiten) * colors[:, :3] + whiten * 1.0
    return ListedColormap(colors, name=f"{name}_pastel_w{whiten:.2f}")

PASTEL_CMAP = make_pastel_cmap(BASE_CMAP_NAME, WHITEN)

EFF = [pe.Stroke(linewidth=5.2 * FONT_SCALE, foreground="white", alpha=0.95), pe.Normal()]
LINE_ID = dict(color="#0b3d91", linestyle="--", linewidth=2.6 * FONT_SCALE, alpha=0.98, zorder=7)
LINE_B  = dict(color="#b22222", linestyle=":",  linewidth=2.0 * FONT_SCALE, alpha=0.95, zorder=7)
CAL_LN  = dict(color="#111111", linewidth=2.4 * FONT_SCALE, marker="o",
               markersize=4.6 * FONT_SCALE, alpha=0.95, zorder=8)

def _to_num(s):
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def prep_xy(df, xcol, ycol):
    x = _to_num(df[xcol]); y = _to_num(df[ycol])
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    return x[m], y[m]

def quantile_window(lx, ly, qlo=0.01, qhi=0.99, pad=0.05):
    xlo, xhi = np.quantile(lx, qlo), np.quantile(lx, qhi)
    ylo, yhi = np.quantile(ly, qlo), np.quantile(ly, qhi)
    dx = (xhi - xlo) * pad
    dy = (yhi - ylo) * pad
    return (xlo - dx, xhi + dx, ylo - dy, yhi + dy)

def draw_ref_lines(ax, lo, hi):
    xx = np.linspace(lo, hi, 320)
    l1, = ax.plot(xx, xx, **LINE_ID)
    l2, = ax.plot(xx, xx + np.log10(2), **LINE_B)
    l3, = ax.plot(xx, xx - np.log10(2), **LINE_B)
    for ln in (l1, l2, l3):
        ln.set_path_effects(EFF)

def calibration_curve(lx, ly, q=12):
    lx = np.asarray(lx); ly = np.asarray(ly)
    if lx.size < 30:
        return pd.DataFrame({"lx_med": [], "ly_med": []})
    q_eff = int(min(q, max(4, lx.size // 10)))
    bins = pd.qcut(lx, q=q_eff, duplicates="drop")
    cal = (
        pd.DataFrame({"lx": lx, "ly": ly, "bin": bins})
        .groupby("bin")
        .agg(lx_med=("lx","median"), ly_med=("ly","median"), n=("lx","size"))
        .reset_index(drop=True)
        .sort_values("lx_med")
    )
    return cal

def compute_hist2d_counts(lx, ly, bins=65, rng=None):
    H, _, _ = np.histogram2d(lx, ly, bins=bins, range=rng)
    return H

def plot_panel(ax, lx, ly, rng, bins, norm):
    h = ax.hist2d(
        lx, ly,
        bins=bins,
        range=rng,
        norm=norm,
        cmap=PASTEL_CMAP,
        alpha=DENSITY_ALPHA
    )
    xlo, xhi = rng[0]
    ylo, yhi = rng[1]
    lo = max(xlo, ylo)
    hi = min(xhi, yhi)
    draw_ref_lines(ax, lo, hi)

    cal = calibration_curve(lx, ly, q=12)
    if len(cal) > 0:
        ln, = ax.plot(cal["lx_med"], cal["ly_med"], **CAL_LN)
        ln.set_path_effects(EFF)

    ax.set_xlim(rng[0])
    ax.set_ylim(rng[1])
    ax.grid(True, alpha=0.18)
    return h[3]

def compute_metrics_log10(x, y):
    lx = np.log10(x)
    ly = np.log10(y)

    err = lx - ly
    sse = np.sum((ly - lx)**2)
    sst = np.sum((ly - ly.mean())**2)
    r2 = 1 - sse / sst if sst > 0 else np.nan
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    corr = np.corrcoef(lx, ly)[0, 1] if len(lx) >= 2 else np.nan
    median_bias_ratio = 10 ** (np.median(err))

    return {
        "n": len(x),
        "r2_log10": r2,
        "mae_log10": mae,
        "rmse_log10": rmse,
        "pearson_r_log10": corr,
        "median_bias_ratio_pred_over_obs": median_bias_ratio,
    }

df = pd.read_csv(IN_CSV, encoding="utf-8-sig").copy()
df = df[df["park_class"].isin(["A", "B"])].copy()

must = ["park_class", "annual_benefit_yen_2019", "upgraded_pred_benefit"]
miss = [c for c in must if c not in df.columns]
if miss:
    raise KeyError(f"Missing columns: {miss}")

PANELS = [
    ("A", "upgraded_pred_benefit", "annual_benefit_yen_2019",
     "Type A annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
    ("B", "upgraded_pred_benefit", "annual_benefit_yen_2019",
     "Type B annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
]

bins = 65
panel_data = []
global_vmax = 1

print("=== BENEFIT FIT METRICS (log10 scale) ===")
for (t, xcol, ycol, title, xlabel, ylabel) in PANELS:
    df_t = df[df["park_class"] == t].copy()
    x, y = prep_xy(df_t, xcol, ycol)

    metrics = compute_metrics_log10(x, y)
    print(f"\nType {t}")
    for k, v in metrics.items():
        print(f"{k}: {v:.6f}" if isinstance(v, (float, np.floating)) else f"{k}: {v}")

    lx = np.log10(x); ly = np.log10(y)
    xlo, xhi, ylo, yhi = quantile_window(lx, ly, 0.01, 0.99, pad=0.05)
    rng = [[xlo, xhi], [ylo, yhi]]

    H = compute_hist2d_counts(lx, ly, bins=bins, rng=rng)
    vmax = int(np.nanmax(H)) if H.size else 1
    global_vmax = max(global_vmax, vmax)

    panel_data.append((lx, ly, rng, title, xlabel, ylabel))

shared_norm = LogNorm(vmin=1, vmax=max(1, global_vmax))

fig, axs = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=False)

fig.subplots_adjust(
    left=0.10,
    right=1.0 - RIGHT_FOR_CBAR - 0.02,
    top=0.92,
    bottom=BOTTOM_FOR_LEGEND,
    wspace=0.35
)

last_mappable = None
for i, (ax, (lx, ly, rng, title, xlabel, ylabel)) in enumerate(zip(axs, panel_data)):
    last_mappable = plot_panel(ax, lx, ly, rng, bins, shared_norm)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    if i == 0:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel("")

cax = fig.add_axes([1.0 - RIGHT_FOR_CBAR, BOTTOM_FOR_LEGEND, 0.015, 0.92 - BOTTOM_FOR_LEGEND])
cb = fig.colorbar(last_mappable, cax=cax)
cb.set_label("Bin count (log scale)")

legend_handles = [
    Line2D([0],[0], **LINE_ID, label="Identity (y = x)"),
    Line2D([0],[0], **LINE_B,  label="×2 / ×0.5 error band"),
    Line2D([0],[0], **CAL_LN,  label="Binned medians"),
]
for h in legend_handles:
    h.set_path_effects(EFF)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.02),
    handlelength=3.0
)

out_png = os.path.join(OUT_DIR, "fig_R3_AB_benefit_only_r2.png")
fig.savefig(out_png, dpi=300, bbox_inches="tight")
print("\nSaved:", out_png)

plt.show()
plt.close(fig)


In [ ]:
# Rebuild the smearing and payback calibrations from the finalized model table.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "ab_modeling_table_r2_rebuilt.csv")

OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_from_rebuilt.csv")

OUT_CSV_SMEAR = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_from_rebuilt.csv")
OUT_CSV_PAYBACKCAL = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_paybackcal_from_rebuilt.csv")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_from_rebuilt_diagnostics.txt")

EPS = 1e-12
CAP_YEARS = 300.0
PAYBACK_HORIZON = 50.0

df = pd.read_csv(IN_CSV, encoding="utf-8-sig", low_memory=False)

print("Loaded:", IN_CSV)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

need = [
    "park_class", "area_m2", "E_pop",
    "one_time_investment_2019",
    "annual_om_cost_2019",
    "annual_benefit_yen_2019",
    "land_price",
]
miss = [c for c in need if c not in df.columns]
if miss:
    raise KeyError(f"Missing required columns in rebuilt table: {miss}")

d = df.copy()

d["abcde"] = d["park_class"].astype(str).str.strip().str.upper()

d["annual_benefit_yen"] = pd.to_numeric(d["annual_benefit_yen_2019"], errors="coerce")
d["annual_maint_yen"]   = pd.to_numeric(d["annual_om_cost_2019"], errors="coerce")

d["land_cost_yen"] = pd.to_numeric(d["one_time_investment_2019"], errors="coerce")

if "land_cost_2019" in d.columns:
    d["land_cost_yen_onlyland"] = pd.to_numeric(d["land_cost_2019"], errors="coerce")
else:
    d["land_cost_yen_onlyland"] = np.nan

d["land_price_yen_per_m2"] = pd.to_numeric(d["land_price"], errors="coerce")
if "maint" in d.columns:
    d["maint_yen_per_m2"] = pd.to_numeric(d["maint"], errors="coerce")
else:
    d["maint_yen_per_m2"] = d["annual_maint_yen"] / d["area_m2"].replace(0, np.nan)

net_obs = d["annual_benefit_yen"] - d["annual_maint_yen"]
d["payback_years"] = np.where(
    np.isfinite(net_obs) & (net_obs > EPS) &
    np.isfinite(d["land_cost_yen"]) & (d["land_cost_yen"] > 0),
    d["land_cost_yen"] / net_obs,
    np.inf
)

d = d[d["abcde"].isin(["A", "B"])].copy()

X_cols = ["land_price_yen_per_m2", "area_m2", "E_pop", "maint_yen_per_m2"]
y_col = "annual_benefit_yen"

for c in X_cols + [y_col, "annual_maint_yen", "land_cost_yen", "payback_years"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

def ols_numpy(X, y):
    n = X.shape[0]
    X1 = np.c_[np.ones(n), X]
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    yhat = X1 @ beta
    resid = y - yhat
    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = 1 - ssr/sst if sst > 0 else np.nan
    return beta, resid, r2, n

def fit_with_smearing(df_ab, type_char):
    sub = df_ab[df_ab["abcde"] == type_char].copy()

    m = np.isfinite(sub[y_col]) & (sub[y_col] > 0)
    for c in X_cols:
        m &= np.isfinite(sub[c]) & (sub[c] > 0)

    sub = sub[m].copy()
    if len(sub) < 50:
        print(f"Type {type_char}: too few samples = {len(sub)}")
        return None

    y = np.log(sub[y_col].clip(lower=EPS)).values
    X = np.column_stack([np.log(sub[c].clip(lower=EPS)).values for c in X_cols])

    beta, resid, r2, n = ols_numpy(X, y)
    smear = float(np.mean(np.exp(resid)))

    print(f"\n=== Type {type_char} benefit model + smearing ===")
    print("n =", n, "| R2 =", float(r2))
    print("smearing S =", smear)
    print("ln(benefit) = a + Σ b ln(x)")
    print("a =", float(beta[0]))
    for j, col in enumerate(X_cols, start=1):
        print(f"b_{col} =", float(beta[j]))

    A0 = float(np.exp(beta[0]) * smear)
    print("\nMultiplicative (smear-corrected) form:")
    print(f"benefit_hat ≈ {A0:.6g}"
          f" * land_price^{beta[1]:.4f}"
          f" * area^{beta[2]:.4f}"
          f" * E_pop^{beta[3]:.4f}"
          f" * maint^{beta[4]:.4f}")

    return beta, smear, r2, n, A0

def predict_benefit_smear(df_ab, type_char, beta, smear):
    idx = (df_ab["abcde"] == type_char)
    sub = df_ab.loc[idx, :].copy()

    m = np.ones(len(sub), dtype=bool)
    for c in X_cols:
        m &= np.isfinite(sub[c].values) & (sub[c].values > 0)

    Xp = np.column_stack([np.log(sub.loc[m, c].clip(lower=EPS)).values for c in X_cols])
    ln_hat = beta[0] + Xp @ beta[1:]
    bhat = np.exp(ln_hat) * smear

    out = np.full(len(sub), np.nan, dtype=float)
    out[m] = bhat
    return idx, out

resA = fit_with_smearing(d, "A")
resB = fit_with_smearing(d, "B")

d["benefit_hat_2step_smear_yen"] = np.nan

if resA is not None:
    betaA, smearA, r2A, nA, A0 = resA
    idxA, outA = predict_benefit_smear(d, "A", betaA, smearA)
    d.loc[idxA, "benefit_hat_2step_smear_yen"] = outA

if resB is not None:
    betaB, smearB, r2B, nB, B0 = resB
    idxB, outB = predict_benefit_smear(d, "B", betaB, smearB)
    d.loc[idxB, "benefit_hat_2step_smear_yen"] = outB

net_hat = d["benefit_hat_2step_smear_yen"] - d["annual_maint_yen"]
d["payback_hat_2step_smear_years"] = np.where(
    np.isfinite(net_hat) & (net_hat > EPS) & np.isfinite(d["land_cost_yen"]) & (d["land_cost_yen"] > 0),
    d["land_cost_yen"] / net_hat,
    np.inf
)

d.to_csv(OUT_CSV_SMEAR, index=False, encoding="utf-8-sig")
print("\nSaved smear-stage file:", OUT_CSV_SMEAR)

base_mask = (
    d["abcde"].isin(["A","B"]) &
    np.isfinite(d["payback_years"]) & (d["payback_years"] > 0) &
    np.isfinite(d["benefit_hat_2step_smear_yen"]) & (d["benefit_hat_2step_smear_yen"] > 0) &
    np.isfinite(d["annual_maint_yen"]) &
    np.isfinite(d["land_cost_yen"]) & (d["land_cost_yen"] > 0)
)

def pred_share_leq50(sub_df, s):
    b = sub_df["benefit_hat_2step_smear_yen"].values
    m = sub_df["annual_maint_yen"].values
    lc = sub_df["land_cost_yen"].values

    net = s * b - m
    T = np.full_like(net, np.inf, dtype=float)
    ok = net > EPS
    T[ok] = lc[ok] / net[ok]
    return float(np.mean(T <= PAYBACK_HORIZON))

def find_s_for_target(sub_df, target, s_lo=0.05, s_hi=5.0, iters=50):
    f_lo = pred_share_leq50(sub_df, s_lo)
    f_hi = pred_share_leq50(sub_df, s_hi)

    if target <= f_lo:
        return s_lo, f_lo, f_hi
    if target >= f_hi:
        return s_hi, f_lo, f_hi

    lo, hi = s_lo, s_hi
    for _ in range(iters):
        mid = (lo + hi) / 2.0
        f_mid = pred_share_leq50(sub_df, mid)
        if f_mid < target:
            lo = mid
        else:
            hi = mid
    s_star = (lo + hi) / 2.0
    return s_star, f_lo, f_hi

scales_paybackcal = {}
for t in ["A","B"]:
    sub = d[base_mask & (d["abcde"] == t)].copy()
    obs_share = float(np.mean(sub["payback_years"].values <= PAYBACK_HORIZON))
    s_star, f_lo, f_hi = find_s_for_target(sub, obs_share)
    pred_star = pred_share_leq50(sub, s_star)
    scales_paybackcal[t] = s_star

    print(f"\n=== Type {t} payback-cal ===")
    print("n =", len(sub))
    print("obs <=50 share =", obs_share)
    print("pred <=50 share @s_lo =", f_lo, "| @s_hi =", f_hi)
    print("calibrated s =", s_star)
    print("pred <=50 share @s* =", pred_star)

d["s_paybackcal"] = d["abcde"].map(scales_paybackcal)
net_paybackcal = d["s_paybackcal"] * d["benefit_hat_2step_smear_yen"] - d["annual_maint_yen"]
d["payback_hat_paybackcal_years"] = np.where(
    net_paybackcal > EPS,
    d["land_cost_yen"] / net_paybackcal,
    np.inf
)

d.to_csv(OUT_CSV_PAYBACKCAL, index=False, encoding="utf-8-sig")
print("Saved paybackcal-stage file:", OUT_CSV_PAYBACKCAL)

def pred_payback(s, b, m, lc):
    """
    Predict payback years, assigning ``CAP_YEARS`` when net benefit is non-positive.
    """
    net = s * b - m
    T = np.full_like(net, CAP_YEARS, dtype=float)
    ok = net > EPS
    T[ok] = lc[ok] / net[ok]
    return np.clip(T, EPS, CAP_YEARS)

def loss_log_mae(s, b, m, lc, Tobs):
    Tpred = pred_payback(s, b, m, lc)
    return float(np.median(np.abs(np.log(Tpred) - np.log(Tobs))))

def search_s_continuous(sub, s_lo=0.05, s_hi=2.0, n_grid=240, n_refine=160):
    b = sub["benefit_hat_2step_smear_yen"].values
    m = sub["annual_maint_yen"].values
    lc = sub["land_cost_yen"].values
    Tobs = np.clip(sub["payback_years"].values, EPS, CAP_YEARS)

    grid = np.logspace(np.log10(s_lo), np.log10(s_hi), n_grid)
    vals = np.array([loss_log_mae(s, b, m, lc, Tobs) for s in grid])
    i0 = int(np.argmin(vals))
    s0 = float(grid[i0])

    lo = max(s_lo, s0 / 1.35)
    hi = min(s_hi, s0 * 1.35)
    grid2 = np.logspace(np.log10(lo), np.log10(hi), n_refine)
    vals2 = np.array([loss_log_mae(s, b, m, lc, Tobs) for s in grid2])
    i1 = int(np.argmin(vals2))
    s1 = float(grid2[i1])
    return s1, float(vals2[i1]), float(s0), float(vals[i0])

scales_contcal = {}
diag_rows = []

for t in ["A","B"]:
    sub = d[base_mask & (d["abcde"] == t)].copy()
    s_star, L_star, s0, L0 = search_s_continuous(sub)
    scales_contcal[t] = s_star

    b = sub["benefit_hat_2step_smear_yen"].values
    m = sub["annual_maint_yen"].values
    lc = sub["land_cost_yen"].values
    Tobs = np.clip(sub["payback_years"].values, EPS, CAP_YEARS)
    Tpred = pred_payback(s_star, b, m, lc)

    ratio = (Tpred / Tobs)

    print(f"\n=== Type {t} contcal ===")
    print(f"n={len(sub)}")
    print(f"best s (continuous) = {s_star:.6f}")
    print(f"log-MAE(median) = {L_star:.6f}")
    print(f"median(Tpred/Tobs) = {float(np.median(ratio)):.4f} | P10/P90 = {float(np.quantile(ratio,0.1)):.4f}/{float(np.quantile(ratio,0.9)):.4f}")

    diag_rows.append({
        "type": t,
        "n": len(sub),
        "s_contcal": s_star,
        "log_mae_median": L_star,
        "median_ratio": float(np.median(ratio)),
        "p10_ratio": float(np.quantile(ratio, 0.1)),
        "p90_ratio": float(np.quantile(ratio, 0.9)),
    })

print("\nContinuous-cal scales:", scales_contcal)

d["s_contcal"] = d["abcde"].map(scales_contcal)
d["benefit_hat_contcal_yen"] = d["benefit_hat_2step_smear_yen"] * d["s_contcal"]

net_cont = d["benefit_hat_contcal_yen"] - d["annual_maint_yen"]
d["payback_hat_contcal_years"] = np.where(
    net_cont > EPS,
    d["land_cost_yen"] / net_cont,
    np.inf
)

sub_all = d[base_mask].copy()
Tobs = np.clip(sub_all["payback_years"].values, EPS, CAP_YEARS)
Tpred = np.clip(sub_all["payback_hat_contcal_years"].replace(np.inf, CAP_YEARS).values, EPS, CAP_YEARS)

overall_log_mae = float(np.median(np.abs(np.log(Tpred) - np.log(Tobs))))
ratio_all = Tpred / Tobs

print("\n=== Overall (A+B) on observed subset ===")
print("median log-abs-error =", overall_log_mae)
print("median(Tpred/Tobs) =", float(np.median(ratio_all)))
print("P10/P90(Tpred/Tobs) =", float(np.quantile(ratio_all,0.1)), "/", float(np.quantile(ratio_all,0.9)))
print(pd.Series(Tpred).describe(percentiles=[0.1,0.5,0.9,0.99]))

keep_cols = list(d.columns)
d.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved final contcal file:", OUT_CSV)

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write(f"INPUT = {IN_CSV}\n")
    f.write(f"OUTPUT = {OUT_CSV}\n")
    f.write(f"CAP_YEARS = {CAP_YEARS}\n")
    f.write(f"PAYBACK_HORIZON = {PAYBACK_HORIZON}\n\n")

    if resA is not None:
        f.write("=== Type A smear model ===\n")
        f.write(f"smear = {smearA}\n")
        f.write(f"R2 = {r2A}\n")
        f.write(f"const_smear = {A0}\n")
        f.write(f"beta = {betaA.tolist()}\n\n")

    if resB is not None:
        f.write("=== Type B smear model ===\n")
        f.write(f"smear = {smearB}\n")
        f.write(f"R2 = {r2B}\n")
        f.write(f"const_smear = {B0}\n")
        f.write(f"beta = {betaB.tolist()}\n\n")

    f.write("=== Paybackcal scales ===\n")
    f.write(str(scales_paybackcal) + "\n\n")

    f.write("=== Contcal scales ===\n")
    f.write(str(scales_contcal) + "\n\n")

    f.write("=== Contcal diagnostics by type ===\n")
    for row in diag_rows:
        f.write(str(row) + "\n")
    f.write("\n")

    f.write("=== Overall (A+B) ===\n")
    f.write(f"median log-abs-error = {overall_log_mae}\n")
    f.write(f"median(Tpred/Tobs) = {float(np.median(ratio_all))}\n")
    f.write(f"P10(Tpred/Tobs) = {float(np.quantile(ratio_all,0.1))}\n")
    f.write(f"P90(Tpred/Tobs) = {float(np.quantile(ratio_all,0.9))}\n")

print("Saved diagnostics:", OUT_DIAG_TXT)


In [ ]:
# Calculate and plot discounted observed and predicted payback.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

OUT_DIR = r"outputs"
IN_CSV  = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_from_rebuilt.csv")
OUT_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_discounted.csv")
OUT_PNG = os.path.join(OUT_DIR, "fig_R3_AB_4panel_row_discounted_from_rebuilt.png")
OUT_TXT = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_discounted_diagnostics.txt")

DISCOUNT_RATE = 0.02
BENEFIT_GROWTH = 0.02
OM_GROWTH = 0.02

EVAL_YEARS = 50
MAX_YEAR_LONG = 500
CAP_YEARS = 300.0
EPS = 1e-12

TITLE_SCALE = 0.82
YTICK_SCALE = 0.82
FONT_SCALE = 1.00
FIGSIZE = (8.27, 3.4)
BOTTOM_FOR_LEGEND = 0.3
RIGHT_FOR_CBAR = 0.035

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11 * FONT_SCALE,
    "axes.titlesize": 12 * FONT_SCALE * TITLE_SCALE,
    "ytick.labelsize": 11 * FONT_SCALE * YTICK_SCALE,
    "xtick.labelsize": 11 * FONT_SCALE * 0.95,
    "axes.labelsize": 11 * FONT_SCALE,
    "legend.fontsize": 10 * FONT_SCALE,
})

BASE_CMAP_NAME = "viridis"
WHITEN = 0.58
DENSITY_ALPHA = 0.95

def make_pastel_cmap(name="viridis", whiten=0.58):
    cmap = plt.get_cmap(name)
    colors = cmap(np.linspace(0, 1, 256))
    colors[:, :3] = (1 - whiten) * colors[:, :3] + whiten * 1.0
    return ListedColormap(colors, name=f"{name}_pastel_w{whiten:.2f}")

PASTEL_CMAP = make_pastel_cmap(BASE_CMAP_NAME, WHITEN)

EFF = [pe.Stroke(linewidth=5.2 * FONT_SCALE, foreground="white", alpha=0.95), pe.Normal()]
LINE_ID = dict(color="#0b3d91", linestyle="--", linewidth=2.6 * FONT_SCALE, alpha=0.98, zorder=7)
LINE_B  = dict(color="#b22222", linestyle=":",  linewidth=2.0 * FONT_SCALE, alpha=0.95, zorder=7)
CAL_LN  = dict(color="#111111", linewidth=2.4 * FONT_SCALE, marker="o",
               markersize=4.6 * FONT_SCALE, alpha=0.95, zorder=8)

def _to_num(s):
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def discounted_payback_with_growth(invest, annual_benefit_0, annual_om_0,
                                   r, g_b, g_om, years=50, max_year_long=500):
    """
    Return full discounted payback year.
    If never pays back within max_year_long, return np.inf.
    """
    if not np.isfinite(invest) or invest <= 0:
        return np.nan
    if not np.isfinite(annual_benefit_0) or annual_benefit_0 <= 0:
        return np.inf
    if not np.isfinite(annual_om_0) or annual_om_0 < 0:
        return np.inf

    cum_pv = 0.0

    for t in range(1, max_year_long + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)
        cum_pv += pv_t
        if cum_pv >= invest:
            return float(t)

    return np.inf

def discounted_payback_vector(invest, benefit0, om0, r, g_b, g_om, years=50, max_year_long=500):
    out = []
    for c0, b0, m0 in zip(invest, benefit0, om0):
        out.append(discounted_payback_with_growth(
            c0, b0, m0, r=r, g_b=g_b, g_om=g_om, years=years, max_year_long=max_year_long
        ))
    return np.array(out, dtype=float)

def prep_xy(df, xcol, ycol, cap=None):
    x = _to_num(df[xcol]); y = _to_num(df[ycol])
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[m]; y = y[m]
    if cap is not None:
        x = np.clip(x, EPS, cap)
        y = np.clip(y, EPS, cap)
    return x, y

def quantile_window(lx, ly, qlo=0.01, qhi=0.99, pad=0.05):
    xlo, xhi = np.quantile(lx, qlo), np.quantile(lx, qhi)
    ylo, yhi = np.quantile(ly, qlo), np.quantile(ly, qhi)
    dx = (xhi - xlo) * pad
    dy = (yhi - ylo) * pad
    return (xlo - dx, xhi + dx, ylo - dy, yhi + dy)

def draw_ref_lines(ax, lo, hi):
    xx = np.linspace(lo, hi, 320)
    l1, = ax.plot(xx, xx, **LINE_ID)
    l2, = ax.plot(xx, xx + np.log10(2), **LINE_B)
    l3, = ax.plot(xx, xx - np.log10(2), **LINE_B)
    for ln in (l1, l2, l3):
        ln.set_path_effects(EFF)

def calibration_curve(lx, ly, q=12):
    lx = np.asarray(lx); ly = np.asarray(ly)
    if lx.size < 30:
        return pd.DataFrame({"lx_med": [], "ly_med": []})
    q_eff = int(min(q, max(4, lx.size // 10)))
    bins = pd.qcut(lx, q=q_eff, duplicates="drop")
    cal = (
        pd.DataFrame({"lx": lx, "ly": ly, "bin": bins})
        .groupby("bin")
        .agg(lx_med=("lx","median"), ly_med=("ly","median"), n=("lx","size"))
        .reset_index(drop=True)
        .sort_values("lx_med")
    )
    return cal

def compute_hist2d_counts(lx, ly, bins=65, rng=None):
    H, _, _ = np.histogram2d(lx, ly, bins=bins, range=rng)
    return H

def plot_panel(ax, lx, ly, rng, bins, norm):
    h = ax.hist2d(
        lx, ly,
        bins=bins,
        range=rng,
        norm=norm,
        cmap=PASTEL_CMAP,
        alpha=DENSITY_ALPHA
    )

    xlo, xhi = rng[0]
    ylo, yhi = rng[1]
    lo = max(xlo, ylo)
    hi = min(xhi, yhi)
    draw_ref_lines(ax, lo, hi)

    cal = calibration_curve(lx, ly, q=12)
    if len(cal) > 0:
        ln, = ax.plot(cal["lx_med"], cal["ly_med"], **CAL_LN)
        ln.set_path_effects(EFF)

    ax.set_xlim(rng[0])
    ax.set_ylim(rng[1])
    ax.grid(True, alpha=0.18)
    return h[3]

def calc_metrics(obs, pred, label):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    m = np.isfinite(obs) & np.isfinite(pred) & (obs > 0) & (pred > 0)
    obs = obs[m]
    pred = pred[m]
    if len(obs) == 0:
        return {"label": label, "n": 0}

    log_obs = np.log(obs)
    log_pred = np.log(pred)
    mae_med = float(np.median(np.abs(log_pred - log_obs)))
    ratio = pred / obs
    return {
        "label": label,
        "n": int(len(obs)),
        "median_log_abs_error": mae_med,
        "median_ratio": float(np.median(ratio)),
        "p10_ratio": float(np.quantile(ratio, 0.1)),
        "p90_ratio": float(np.quantile(ratio, 0.9)),
    }

df = pd.read_csv(IN_CSV, encoding="utf-8-sig", low_memory=False)
df = df[df["abcde"].isin(["A", "B"])].copy()

need = [
    "abcde",
    "annual_benefit_yen",
    "benefit_hat_contcal_yen",
    "annual_maint_yen",
    "land_cost_yen",
]
miss = [c for c in need if c not in df.columns]
if miss:
    raise KeyError(f"Missing columns in {IN_CSV}: {miss}")

for c in need[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["payback_years_discounted"] = discounted_payback_vector(
    invest=df["land_cost_yen"].values,
    benefit0=df["annual_benefit_yen"].values,
    om0=df["annual_maint_yen"].values,
    r=DISCOUNT_RATE,
    g_b=BENEFIT_GROWTH,
    g_om=OM_GROWTH,
    years=EVAL_YEARS,
    max_year_long=MAX_YEAR_LONG
)

df["payback_hat_contcal_years_discounted"] = discounted_payback_vector(
    invest=df["land_cost_yen"].values,
    benefit0=df["benefit_hat_contcal_yen"].values,
    om0=df["annual_maint_yen"].values,
    r=DISCOUNT_RATE,
    g_b=BENEFIT_GROWTH,
    g_om=OM_GROWTH,
    years=EVAL_YEARS,
    max_year_long=MAX_YEAR_LONG
)

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved discounted table:", OUT_CSV)

diag = []
for t in ["A", "B"]:
    sub = df[df["abcde"] == t].copy()
    diag.append(calc_metrics(sub["annual_benefit_yen"], sub["benefit_hat_contcal_yen"], f"Type {t} benefit"))
    diag.append(calc_metrics(sub["payback_years_discounted"], sub["payback_hat_contcal_years_discounted"], f"Type {t} discounted payback"))

with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write(f"INPUT = {IN_CSV}\n")
    f.write(f"OUTPUT = {OUT_CSV}\n")
    f.write(f"DISCOUNT_RATE = {DISCOUNT_RATE}\n")
    f.write(f"BENEFIT_GROWTH = {BENEFIT_GROWTH}\n")
    f.write(f"OM_GROWTH = {OM_GROWTH}\n")
    f.write(f"EVAL_YEARS = {EVAL_YEARS}\n")
    f.write(f"MAX_YEAR_LONG = {MAX_YEAR_LONG}\n\n")
    for row in diag:
        f.write(str(row) + "\n")

print("Saved diagnostics:", OUT_TXT)

PANELS = [
    ("A", "payback", "payback_hat_contcal_years_discounted", "payback_years_discounted", CAP_YEARS,
     "Type A discounted payback", "Predicted (log$_{10}$ years)", "Observed (log$_{10}$ years)"),
    ("A", "benefit", "benefit_hat_contcal_yen", "annual_benefit_yen", None,
     "Type A annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
    ("B", "payback", "payback_hat_contcal_years_discounted", "payback_years_discounted", CAP_YEARS,
     "Type B discounted payback", "Predicted (log$_{10}$ years)", "Observed (log$_{10}$ years)"),
    ("B", "benefit", "benefit_hat_contcal_yen", "annual_benefit_yen", None,
     "Type B annual health benefit", "Predicted (log$_{10}$ yen/yr)", "Observed (log$_{10}$ yen/yr)"),
]

bins = 65
panel_data = []
global_vmax = 1

for (t, kind, xcol, ycol, cap, title, xlabel, ylabel) in PANELS:
    df_t = df[df["abcde"] == t].copy()
    x, y = prep_xy(df_t, xcol, ycol, cap=cap)
    lx = np.log10(x)
    ly = np.log10(y)
    xlo, xhi, ylo, yhi = quantile_window(lx, ly, 0.01, 0.99, pad=0.05)
    rng = [[xlo, xhi], [ylo, yhi]]

    H = compute_hist2d_counts(lx, ly, bins=bins, rng=rng)
    vmax = int(np.nanmax(H)) if H.size else 1
    global_vmax = max(global_vmax, vmax)

    panel_data.append((lx, ly, rng, title, xlabel, ylabel))

shared_norm = LogNorm(vmin=1, vmax=max(1, global_vmax))

fig, axs = plt.subplots(1, 4, figsize=FIGSIZE, constrained_layout=False)
fig.subplots_adjust(
    left=0.06,
    right=1.0 - RIGHT_FOR_CBAR - 0.02,
    top=0.92,
    bottom=BOTTOM_FOR_LEGEND,
    wspace=0.32
)

last_mappable = None
for i, (ax, (lx, ly, rng, title, xlabel, ylabel)) in enumerate(zip(axs, panel_data)):
    last_mappable = plot_panel(ax, lx, ly, rng, bins=bins, norm=shared_norm)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    if i == 0:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel("")

cax = fig.add_axes([1.0 - RIGHT_FOR_CBAR, BOTTOM_FOR_LEGEND, 0.012, 0.92 - BOTTOM_FOR_LEGEND])
cb = fig.colorbar(last_mappable, cax=cax)
cb.set_label("Bin count (log scale)")

legend_handles = [
    Line2D([0],[0], **LINE_ID, label="Identity (y = x)"),
    Line2D([0],[0], **LINE_B,  label="×2 / ×0.5 error band"),
    Line2D([0],[0], **CAL_LN,  label="Binned medians"),
]
for h in legend_handles:
    h.set_path_effects(EFF)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.02),
    handlelength=3.0
)

fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
print("Saved figure:", OUT_PNG)

plt.show()
plt.close(fig)


In [ ]:
# Calculate analytical payback elasticities for the finalized benefit equations.
import os
import numpy as np
import pandas as pd

OUT_DIR = r"outputs"
IN_CSV = os.path.join(OUT_DIR, "park_result3_AB_twostep_payback_hat_smear_contcal_discounted.csv")

OUT_BASELINE_CSV = os.path.join(OUT_DIR, "sensitivity_discounted_payback_baseline_corrected.csv")
OUT_ELASTICITY_CSV = os.path.join(OUT_DIR, "sensitivity_discounted_payback_elasticity_corrected.csv")
OUT_TXT = os.path.join(OUT_DIR, "sensitivity_discounted_payback_summary_corrected.txt")

r = 0.02
g_B = 0.02
g_OM = 0.02

AREA_BASE = {
    "A": 1800.0,
    "B": 11000.0,
}

PARAMS = {
    "A": {
        "kappa": 0.0347096465,
        "b_land": 0.8128914210,
        "b_area": 0.9656027530,
        "b_epop": 0.1059226996,
        "b_maint": 0.2507624657,
    },
    "B": {
        "kappa": 0.3369975754,
        "b_land": 0.7718787964,
        "b_area": 0.7215718933,
        "b_epop": 0.0950937969,
        "b_maint": 0.2940707200,
    }
}

def finite_median(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s[np.isfinite(s)]
    return float(np.median(s)) if len(s) else np.nan

def calc_B(type_char, land_price, area, Epop, maint):
    p = PARAMS[type_char]
    return (
        p["kappa"]
        * (land_price ** p["b_land"])
        * (area ** p["b_area"])
        * (Epop ** p["b_epop"])
        * (maint ** p["b_maint"])
    )

df = pd.read_csv(IN_CSV, encoding="utf-8-sig", low_memory=False)
df["abcde"] = df["abcde"].astype(str).str.strip().str.upper()

need = ["abcde", "land_price_yen_per_m2", "E_pop", "maint_yen_per_m2", "area_m2", "land_cost_yen"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise KeyError(f"Missing columns: {miss}")

for c in need[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["k_con_inferred"] = df["land_cost_yen"] / df["area_m2"] - df["land_price_yen_per_m2"]

baseline_rows = []
elasticity_rows = []

for t in ["A", "B"]:
    sub = df[df["abcde"] == t].copy()

    land_price0 = finite_median(sub["land_price_yen_per_m2"])
    Epop0 = finite_median(sub["E_pop"])
    maint0 = finite_median(sub["maint_yen_per_m2"])
    area0 = AREA_BASE[t]
    kcon0 = finite_median(sub["k_con_inferred"])

    p = PARAMS[t]

    B0 = calc_B(t, land_price0, area0, Epop0, maint0)
    OM0 = area0 * maint0
    N0 = B0 - OM0
    C0 = area0 * (land_price0 + kcon0)

    Tstar = (1 + r) * C0 / N0 if N0 > 0 else np.inf

    s_land = land_price0 / (land_price0 + kcon0)

    elas_land = s_land - (p["b_land"] * B0 / N0)
    elas_area = 1.0 - ((p["b_area"] * B0 - OM0) / N0)
    elas_epop = - (p["b_epop"] * B0 / N0)
    elas_maint = (OM0 - p["b_maint"] * B0) / N0

    baseline_rows.append({
        "type": t,
        "land_price_base": land_price0,
        "area_base": area0,
        "E_pop_base": Epop0,
        "maint_base": maint0,
        "k_con_base": kcon0,
        "s_land_base": s_land,
        "B0_base": B0,
        "OM0_base": OM0,
        "N0_base": N0,
        "C0_base": C0,
        "Tstar_discounted_base": Tstar,
        "is_feasible": bool(np.isfinite(Tstar))
    })

    elasticity_dict = {
        "land_price": elas_land,
        "area": elas_area,
        "E_pop": elas_epop,
        "maint": elas_maint,
    }

    for var, elas in elasticity_dict.items():
        effect_10pct = ((1.1 ** elas) - 1.0) * 100.0 if np.isfinite(elas) else np.nan
        direction = "increase" if elas < 0 else "decrease"
        elasticity_rows.append({
            "type": t,
            "variable": var,
            "elasticity_lnTstar_lnx": elas,
            "effect_of_plus10pct_on_Tstar_pct": effect_10pct,
            "beneficial_direction": direction,
            "Tstar_base": Tstar
        })

baseline_df = pd.DataFrame(baseline_rows)
elasticity_df = pd.DataFrame(elasticity_rows)

elasticity_df["abs_elasticity"] = elasticity_df["elasticity_lnTstar_lnx"].abs()
elasticity_df["rank_within_type"] = (
    elasticity_df.groupby("type")["abs_elasticity"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

baseline_df.to_csv(OUT_BASELINE_CSV, index=False, encoding="utf-8-sig")
elasticity_df.to_csv(OUT_ELASTICITY_CSV, index=False, encoding="utf-8-sig")

with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("=== SETTINGS ===\n")
    f.write(f"INPUT = {IN_CSV}\n")
    f.write(f"r = {r}\n")
    f.write(f"g_B = {g_B}\n")
    f.write(f"g_OM = {g_OM}\n\n")

    f.write("=== BASELINE ===\n")
    f.write(baseline_df.to_string(index=False))
    f.write("\n\n=== ELASTICITY ===\n")
    f.write(elasticity_df.to_string(index=False))

print("Saved baseline:", OUT_BASELINE_CSV)
print("Saved elasticity:", OUT_ELASTICITY_CSV)
print("Saved txt:", OUT_TXT)

print("\n=== BASELINE ===")
print(baseline_df)

print("\n=== ELASTICITY ===")
print(elasticity_df)


In [ ]:
# Generate finalized candidate-site payback maps.
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from matplotlib.colors import Normalize, LinearSegmentedColormap

FONT_SCALE = 2.0
plt.rcParams.update({
    "font.size": 10 * FONT_SCALE,
    "axes.titlesize": 12 * FONT_SCALE,
    "axes.labelsize": 10 * FONT_SCALE,
    "xtick.labelsize": 9 * FONT_SCALE,
    "ytick.labelsize": 9 * FONT_SCALE,
})

BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"
DID_SHP  = r"data/public/population/population_districts_2015.shp"
PARKS_CSV = r"outputs/park_result3_AB_twostep_payback_hat_smear_contcal_discounted.csv"
LANDPRICE_SHP = r"data/public/land_prices/official_land_prices.shp"

GRID_STEP_M = 250
R_M = 7000.0
EPS = 1e-9
CAP_YEARS = 300.0

POP_COL = "A16_005"
DID_CRS_FALLBACK = "EPSG:6668"
LP_CRS_FALLBACK  = "EPSG:6668"
LP_VALUE_COL = "L01_008"

A_HA, B_HA = 0.18, 0.90
A_AREA_M2 = A_HA * 10_000.0
B_AREA_M2 = B_HA * 10_000.0

MARKER_EXTRA_SHRINK = 0.20
S_MIN, S_MAX = 1.0, 20.0

Y_5, Y_20, Y_50 = 0.25, 0.45, 0.82

PAYBACK_MODE = "discounted"

DISCOUNT_RATE = 0.02
BENEFIT_GROWTH = 0.02
OM_GROWTH = 0.02
MAX_YEAR_LONG = 500

TARGET_TOTAL_POP = 43_000_000

BENEFIT_PARAMS = {
    "A": dict(
        kappa=0.0347096465,
        a_lp=0.8128914210,
        a_area=0.9656027530,
        a_epop=0.1059226996,
        a_maint=0.2507624657
    ),
    "B": dict(
        kappa=0.3369975754,
        a_lp=0.7718787964,
        a_area=0.7215718933,
        a_epop=0.0950937969,
        a_maint=0.2940707200
    ),
}

PARAM_LOGESIGMA_BETA = {
    "A": (11.7571, -0.4160),
    "B": (11.9702, -0.4193),
    "C": (12.3210, -0.4217),
    "D": (12.2487, -0.3829),
    "E": (12.7412, -0.4308),
}
SIGMA = {k: float(np.exp(v[0])) for k, v in PARAM_LOGESIGMA_BETA.items()}
BETA  = {k: float(v[1]) for k, v in PARAM_LOGESIGMA_BETA.items()}

def safe_read_shp(path, columns_needed=None, crs_fallback=None):
    kwargs = {}
    if columns_needed is not None:
        kwargs["columns"] = columns_needed
    try:
        gdf = gpd.read_file(path, engine="pyogrio", **kwargs)
    except Exception as e1:
        print("[safe_read_shp] pyogrio failed -> fallback to fiona:", repr(e1))
        gdf = gpd.read_file(path, engine="fiona", **kwargs)
    if gdf.crs is None and crs_fallback is not None:
        gdf = gdf.set_crs(crs_fallback, allow_override=True)
    return gdf

class PiecewiseNorm(Normalize):
    def __init__(self, x, y, clip=False):
        super().__init__(vmin=min(x), vmax=max(x), clip=clip)
        self.x = np.asarray(x, float)
        self.y = np.asarray(y, float)
        if not (np.all(np.diff(self.x) > 0) and np.all(np.diff(self.y) > 0)):
            raise ValueError("x and y must be strictly increasing.")
    def __call__(self, value, clip=None):
        v = np.asarray(value, float)
        if (clip is True) or (self.clip is True):
            v = np.clip(v, self.vmin, self.vmax)
        return np.interp(v, self.x, self.y)
    def inverse(self, value):
        v = np.asarray(value, float)
        return np.interp(v, self.y, self.x)

def build_segment_cmap(y5=Y_5, y20=Y_20, y50=Y_50):
    stops = [
        (0.00, (0.90, 1.00, 0.90)),
        (y5,   (0.10, 0.65, 0.20)),
        (y5,   (0.88, 1.00, 1.00)),
        (y20,  (0.00, 0.70, 0.75)),
        (y20,  (0.90, 0.95, 1.00)),
        (y50,  (0.10, 0.30, 0.85)),
        (y50,  (1.00, 0.92, 0.92)),
        (1.00, (0.55, 0.00, 0.00)),
    ]
    return LinearSegmentedColormap.from_list("payback_piecewise", stops, N=256)

def robust_median_positive(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s[np.isfinite(s) & (s > 0)]
    return float(np.median(s)) if len(s) else np.nan

def benefit_hat(type_letter, land_price, area_m2, epop, maint_yen_per_m2_yr):
    p = BENEFIT_PARAMS[type_letter]
    return (
        p["kappa"]
        * np.power(np.maximum(land_price, EPS), p["a_lp"])
        * np.power(np.maximum(area_m2, EPS), p["a_area"])
        * np.power(np.maximum(epop, 1.0), p["a_epop"])
        * np.power(np.maximum(maint_yen_per_m2_yr, EPS), p["a_maint"])
    )

def discounted_payback_with_growth(C0, B0, OM0, r=0.02, g_b=0.02, g_om=0.02, max_year_long=500):
    if not np.isfinite(C0) or C0 <= 0:
        return np.nan
    if not np.isfinite(B0) or B0 <= 0:
        return np.inf
    if not np.isfinite(OM0) or OM0 < 0:
        return np.inf

    cum_pv = 0.0
    for t in range(1, max_year_long + 1):
        Bt = B0 * ((1 + g_b) ** (t - 1))
        OMt = OM0 * ((1 + g_om) ** (t - 1))
        net_t = Bt - OMt
        pv_t = net_t / ((1 + r) ** t)
        cum_pv += pv_t
        if cum_pv >= C0:
            return float(t)
    return np.inf

def payback_hat_simple(C0, benefit_yen_yr, annual_maint_yen):
    net = benefit_yen_yr - annual_maint_yen
    T = np.full_like(net, np.nan, dtype=float)
    ok = net > EPS
    T[ok] = C0[ok] / net[ok]
    T = np.where(np.isfinite(T), np.clip(T, 0.0, CAP_YEARS), np.nan)
    return T, ok

def payback_hat_discounted(C0, benefit_yen_yr, annual_maint_yen,
                           r=0.02, g_b=0.02, g_om=0.02, max_year_long=500):
    out = np.full_like(benefit_yen_yr, np.nan, dtype=float)
    ok = np.isfinite(C0) & np.isfinite(benefit_yen_yr) & np.isfinite(annual_maint_yen) & (C0 > 0)
    idx = np.where(ok)[0]
    for i in idx:
        out[i] = discounted_payback_with_growth(
            C0[i], benefit_yen_yr[i], annual_maint_yen[i],
            r=r, g_b=g_b, g_om=g_om, max_year_long=max_year_long
        )
    out = np.where(np.isfinite(out), np.minimum(out, CAP_YEARS), np.nan)
    feasible = np.isfinite(out)
    return out, feasible

base = safe_read_shp(BASE_SHP, crs_fallback=DID_CRS_FALLBACK)
base = base.reset_index(drop=True)
base_geom = base.unary_union

did = safe_read_shp(DID_SHP, columns_needed=[POP_COL], crs_fallback=DID_CRS_FALLBACK)
did = did.to_crs(base.crs).reset_index(drop=True)

clip_gdf = gpd.GeoDataFrame(geometry=[base_geom], crs=base.crs)
did_clip = gpd.overlay(did, clip_gdf, how="intersection", keep_geom_type=False).reset_index(drop=True)

did_clip[POP_COL] = pd.to_numeric(did_clip[POP_COL], errors="coerce").fillna(0.0)
pop_pts = did_clip[did_clip[POP_COL] > 0].copy()
pop_pts["geometry"] = pop_pts.geometry.representative_point()
pop_pts = pop_pts.reset_index(drop=True)

base_ll = gpd.GeoSeries([base_geom], crs=base.crs).to_crs("EPSG:4326").iloc[0]
cent = base_ll.centroid
lon, lat = cent.x, cent.y
utm_zone = int((lon + 180) // 6) + 1
utm_epsg = 32600 + utm_zone if lat >= 0 else 32700 + utm_zone
PROJ_CRS = f"EPSG:{utm_epsg}"

base_m = gpd.GeoSeries([base_geom], crs=base.crs).to_crs(PROJ_CRS).iloc[0]
boundary = gpd.GeoSeries([base_m], crs=PROJ_CRS).boundary

pop_pts_m = pop_pts.to_crs(PROJ_CRS).reset_index(drop=True)
pop_xy = np.column_stack([pop_pts_m.geometry.x.to_numpy(), pop_pts_m.geometry.y.to_numpy()]).astype(float)
pop_w  = pd.to_numeric(pop_pts_m[POP_COL], errors="coerce").fillna(0.0).to_numpy(dtype=float).copy()

print(f"[pop points] n={len(pop_pts_m)} | pop sum(raw)={pop_w.sum():,.2f}")
print("[projection CRS]", PROJ_CRS)

cur_pop = float(np.nansum(pop_w))
# District population weights are rescaled to the stated study-area population total.
POP_SCALE = TARGET_TOTAL_POP / cur_pop if cur_pop > 0 else 1.0
print(f"[POP_SCALE] cur_pop={cur_pop:,.2f} | target={TARGET_TOTAL_POP:,.0f} | scale={POP_SCALE:.4f}")
pop_w = pop_w * POP_SCALE
print(f"[pop sum(scaled)] {float(np.nansum(pop_w)):,.2f}")

parks = pd.read_csv(PARKS_CSV, encoding="utf-8-sig", low_memory=False)
parks = parks.copy()

if "abcde" not in parks.columns:
    raise KeyError("Cannot find abcde in PARKS_CSV.")
parks["abcde"] = parks["abcde"].astype(str).str.strip().str.upper()

lon_candidates = [c for c in ["Lng","lon","longitude","Lon","LONG"] if c in parks.columns]
lat_candidates = [c for c in ["Lat","lat","latitude","LAT"] if c in parks.columns]
if not lon_candidates or not lat_candidates:
    raise KeyError("PARKS_CSV missing lon/lat columns.")
LON_COL, LAT_COL = lon_candidates[0], lat_candidates[0]

area_candidates = [c for c in ["area_m2","area","Area m2","Area_m2"] if c in parks.columns]
if not area_candidates:
    raise KeyError("PARKS_CSV missing area column.")
AREA_COL = area_candidates[0]

for c in [LON_COL, LAT_COL, AREA_COL]:
    parks[c] = pd.to_numeric(parks[c], errors="coerce")

parks = parks[np.isfinite(parks[LON_COL]) & np.isfinite(parks[LAT_COL])].copy()
parks = parks.reset_index(drop=True)

if "maint_yen_per_m2" in parks.columns:
    parks["maint_yen_per_m2_use"] = pd.to_numeric(parks["maint_yen_per_m2"], errors="coerce")
elif "maint" in parks.columns:
    parks["maint_yen_per_m2_use"] = pd.to_numeric(parks["maint"], errors="coerce")
elif "annual_om_cost_2019" in parks.columns:
    parks["maint_yen_per_m2_use"] = (
        pd.to_numeric(parks["annual_om_cost_2019"], errors="coerce")
        / pd.to_numeric(parks[AREA_COL], errors="coerce").replace(0, np.nan)
    )
elif "annual_maint_yen" in parks.columns:
    parks["maint_yen_per_m2_use"] = (
        pd.to_numeric(parks["annual_maint_yen"], errors="coerce")
        / pd.to_numeric(parks[AREA_COL], errors="coerce").replace(0, np.nan)
    )
else:
    raise KeyError("PARKS_CSV missing maintenance information.")

if "land_price_yen_per_m2" in parks.columns:
    parks["land_price_yen_per_m2"] = pd.to_numeric(parks["land_price_yen_per_m2"], errors="coerce")
elif "land_price" in parks.columns:
    parks["land_price_yen_per_m2"] = pd.to_numeric(parks["land_price"], errors="coerce")
else:
    raise KeyError("PARKS_CSV missing land price per m2 information.")

if "land_cost_yen" in parks.columns:
    parks["land_cost_yen"] = pd.to_numeric(parks["land_cost_yen"], errors="coerce")
elif "one_time_investment_2019" in parks.columns:
    parks["land_cost_yen"] = pd.to_numeric(parks["one_time_investment_2019"], errors="coerce")
else:
    raise KeyError("PARKS_CSV missing total one-off cost information.")

gparks = gpd.GeoDataFrame(
    parks,
    geometry=gpd.points_from_xy(parks[LON_COL], parks[LAT_COL]),
    crs="EPSG:4326"
).to_crs(PROJ_CRS).reset_index(drop=True)

MAINT_A = robust_median_positive(gparks.loc[gparks["abcde"]=="A", "maint_yen_per_m2_use"])
MAINT_B = robust_median_positive(gparks.loc[gparks["abcde"]=="B", "maint_yen_per_m2_use"])
print(f"[maint ref] A median={MAINT_A:.4g} | B median={MAINT_B:.4g}")
if not (np.isfinite(MAINT_A) and np.isfinite(MAINT_B)):
    raise ValueError("MAINT_A/MAINT_B is NaN.")

# Construction cost per square metre is inferred as total unit investment minus land price.
gparks["k_con_inferred"] = (
    pd.to_numeric(gparks["land_cost_yen"], errors="coerce")
    / pd.to_numeric(gparks[AREA_COL], errors="coerce").replace(0, np.nan)
    - pd.to_numeric(gparks["land_price_yen_per_m2"], errors="coerce")
)

KCON_A = robust_median_positive(gparks.loc[gparks["abcde"]=="A", "k_con_inferred"])
KCON_B = robust_median_positive(gparks.loc[gparks["abcde"]=="B", "k_con_inferred"])
KCON_ALL = robust_median_positive(gparks["k_con_inferred"])
if not np.isfinite(KCON_A):
    KCON_A = KCON_ALL
if not np.isfinite(KCON_B):
    KCON_B = KCON_ALL
print(f"[k_con ref] A median={KCON_A:.4f} | B median={KCON_B:.4f}")

park_xy = np.column_stack([gparks.geometry.x.to_numpy(), gparks.geometry.y.to_numpy()]).astype(float)
park_type = gparks["abcde"].to_numpy(dtype=object)

sigma_p = np.array([SIGMA.get(t, np.nan) for t in park_type], dtype=float)
beta_p  = np.array([BETA.get(t, np.nan)  for t in park_type], dtype=float)
okp = np.isfinite(sigma_p) & np.isfinite(beta_p)

park_xy2 = park_xy[okp].copy()
sigma_p2 = sigma_p[okp].copy()
beta_p2  = beta_p[okp].copy()

dx = pop_xy[:, 0:1] - park_xy2[None, :, 0]
dy = pop_xy[:, 1:2] - park_xy2[None, :, 1]
dist = np.sqrt(dx*dx + dy*dy)

mask = dist <= R_M
dist_safe = np.maximum(dist, 1.0)

W = sigma_p2[None, :] * np.power(dist_safe, beta_p2[None, :])
W[~mask] = 0.0
sum_w_existing = W.sum(axis=1)

print("[sum_w_existing] min/median/max:",
      float(np.min(sum_w_existing)), float(np.median(sum_w_existing)), float(np.max(sum_w_existing)))

minx, miny, maxx, maxy = base_m.bounds
xs = np.arange(minx, maxx + GRID_STEP_M, GRID_STEP_M)
ys = np.arange(miny, maxy + GRID_STEP_M, GRID_STEP_M)
XX, YY = np.meshgrid(xs, ys)
grid_xy = np.column_stack([XX.ravel(), YY.ravel()]).astype(float)

grid_gdf = gpd.GeoDataFrame(geometry=[Point(xy) for xy in grid_xy], crs=PROJ_CRS)
inside = grid_gdf.within(base_m).to_numpy()
grid_in = grid_xy[inside].copy()
grid_gdf_in = grid_gdf.loc[inside].copy().reset_index(drop=True)

print(f"[grid] total={len(grid_xy):,} | inside={len(grid_in):,} | step={GRID_STEP_M}m")

lp = safe_read_shp(LANDPRICE_SHP, crs_fallback=LP_CRS_FALLBACK).to_crs(PROJ_CRS).reset_index(drop=True)
if LP_VALUE_COL not in lp.columns:
    raise KeyError(f"LP_VALUE_COL={LP_VALUE_COL} not found.")
lp[LP_VALUE_COL] = pd.to_numeric(lp[LP_VALUE_COL], errors="coerce")

joined = gpd.sjoin(
    grid_gdf_in.reset_index(drop=True),
    lp[[LP_VALUE_COL, "geometry"]].reset_index(drop=True),
    how="left",
    predicate="within"
)
joined = joined.reset_index(drop=True)

lp_grid = np.array(joined[LP_VALUE_COL], dtype=float)

miss = ~np.isfinite(lp_grid)
if miss.any():
    nearest = gpd.sjoin_nearest(
        grid_gdf_in.loc[miss].reset_index(drop=True),
        lp[[LP_VALUE_COL, "geometry"]].reset_index(drop=True),
        how="left"
    ).reset_index(drop=True)
    nearest_vals = np.array(nearest[LP_VALUE_COL], dtype=float)
    if len(nearest_vals) != int(miss.sum()):
        raise ValueError("Nearest land-price join length mismatch.")
    lp_grid[miss] = nearest_vals

lp_grid = np.where(np.isfinite(lp_grid) & (lp_grid > 0), lp_grid, np.nan)
valid_lp = np.isfinite(lp_grid)

print("[land price] finite share:", float(np.mean(valid_lp)))
print("[land price] median (finite):", float(np.nanmedian(lp_grid)))

def compute_epop(grid_in_xy, type_letter, chunk=6000):
    sig = SIGMA[type_letter]
    bet = BETA[type_letter]
    out = np.zeros(len(grid_in_xy), dtype=float)

    for s in range(0, len(grid_in_xy), chunk):
        e = min(len(grid_in_xy), s + chunk)
        g = grid_in_xy[s:e]

        dx = g[:, 0:1] - pop_xy[None, :, 0]
        dy = g[:, 1:2] - pop_xy[None, :, 1]
        d = np.sqrt(dx*dx + dy*dy)

        m = d <= R_M
        d_safe = np.maximum(d, 1.0)
        w_c = sig * np.power(d_safe, bet)
        w_c[~m] = 0.0

        denom = sum_w_existing[None, :] + w_c
        share = np.where(denom > 0, w_c / denom, 0.0)
        out[s:e] = (share * pop_w[None, :]).sum(axis=1)

    return out

Epop_A = compute_epop(grid_in, "A")
Epop_B = compute_epop(grid_in, "B")

TA = np.full(len(grid_in), np.nan, dtype=float)
TB = np.full(len(grid_in), np.nan, dtype=float)

benA = benefit_hat("A", lp_grid[valid_lp], A_AREA_M2, Epop_A[valid_lp], MAINT_A)
benB = benefit_hat("B", lp_grid[valid_lp], B_AREA_M2, Epop_B[valid_lp], MAINT_B)

C0A = A_AREA_M2 * (lp_grid[valid_lp] + KCON_A)
C0B = B_AREA_M2 * (lp_grid[valid_lp] + KCON_B)

OMA = np.full_like(benA, A_AREA_M2 * MAINT_A, dtype=float)
OMB = np.full_like(benB, B_AREA_M2 * MAINT_B, dtype=float)

if PAYBACK_MODE == "simple":
    TA_valid, okA = payback_hat_simple(C0A, benA, OMA)
    TB_valid, okB = payback_hat_simple(C0B, benB, OMB)
else:
    TA_valid, okA = payback_hat_discounted(
        C0A, benA, OMA,
        r=DISCOUNT_RATE, g_b=BENEFIT_GROWTH, g_om=OM_GROWTH,
        max_year_long=MAX_YEAR_LONG
    )
    TB_valid, okB = payback_hat_discounted(
        C0B, benB, OMB,
        r=DISCOUNT_RATE, g_b=BENEFIT_GROWTH, g_om=OM_GROWTH,
        max_year_long=MAX_YEAR_LONG
    )

TA[valid_lp] = TA_valid
TB[valid_lp] = TB_valid

TA_full = np.full(len(grid_xy), np.nan, dtype=float)
TB_full = np.full(len(grid_xy), np.nan, dtype=float)
TA_full[inside] = TA
TB_full[inside] = TB

TA_img = TA_full.reshape(YY.shape)
TB_img = TB_full.reshape(YY.shape)

NORM = PiecewiseNorm(
    x=[0, 5, 20, 50, 300],
    y=[0.00, Y_5, Y_20, Y_50, 1.00],
    clip=True
)
CMAP = build_segment_cmap(Y_5, Y_20, Y_50).copy()
CMAP.set_bad("white")

area_vals = pd.to_numeric(gparks[AREA_COL], errors="coerce").to_numpy(dtype=float)
area_vals = np.where(np.isfinite(area_vals) & (area_vals > 0), area_vals, np.nan)
sqrtA = np.sqrt(area_vals)
s_raw = (sqrtA / 25.0)**2
s = np.clip(s_raw * 100.0 * MARKER_EXTRA_SHRINK, S_MIN, S_MAX)

def draw_map(ax, img, title):
    im = ax.imshow(
        img,
        origin="lower",
        extent=[minx, maxx, miny, maxy],
        cmap=CMAP,
        norm=NORM,
        interpolation="bilinear"
    )
    boundary.plot(ax=ax, linewidth=1.1, color="black")
    ax.scatter(
        gparks.geometry.x.to_numpy(),
        gparks.geometry.y.to_numpy(),
        s=s,
        facecolors="none",
        edgecolors="black",
        linewidths=0.30,
        alpha=0.30,
        zorder=3
    )
    ax.set_title(title, pad=10 * FONT_SCALE)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_axis_off()
    return im

fig, axs = plt.subplots(1, 2, figsize=(18.0, 7.2), constrained_layout=True)

title_suffix = "discounted payback" if PAYBACK_MODE == "discounted" else "simple payback"
im0 = draw_map(axs[0], TA_img, f"Type A ({A_HA:.2f} ha), {title_suffix}")
im1 = draw_map(axs[1], TB_img, f"Type B ({B_HA:.2f} ha), {title_suffix}")

cb = fig.colorbar(im0, ax=axs, fraction=0.046, pad=0.02)
cb.set_label("Predicted payback time (years)", fontsize=10 * FONT_SCALE)
cb.set_ticks([0, 5, 20, 50, 100, 200, 300])
cb.ax.tick_params(labelsize=9 * FONT_SCALE)

plt.show()

def diagnose(arr, name):
    a = arr.copy()
    finite = np.isfinite(a)
    print(f"\n[{name}] finite share (inside grid) = {finite.mean()*100:.2f}% | NaN(white)={100 - finite.mean()*100:.2f}%")

diagnose(TA, "Type A")
diagnose(TB, "Type B")

print("\n[feasibility among land-price-valid points]")
print(f"  Type A: feasible share ≈ {float(np.mean(okA))*100:.2f}%")
print(f"  Type B: feasible share ≈ {float(np.mean(okB))*100:.2f}%")


In [ ]:
# Summarize finalized payback outcomes by park type.
import pandas as pd
import numpy as np

def classify_payback(T, feasible_flag):
    cat = np.full(len(T), "invalid", dtype=object)

    infeasible = ~feasible_flag
    finite = feasible_flag & np.isfinite(T)

    cat[infeasible] = "infeasible"
    cat[finite & (T < 5)] = "<5"
    cat[finite & (T >= 5) & (T < 20)] = "5-20"
    cat[finite & (T >= 20) & (T <= 50)] = "20-50"
    cat[finite & (T > 50)] = ">50"
    return cat

def summarize_type(T_all, valid_lp_mask, feasible_flag, type_name):
    valid_lp_mask = np.asarray(valid_lp_mask, dtype=bool)
    feasible_flag = np.asarray(feasible_flag, dtype=bool)
    T_all = np.asarray(T_all, dtype=float)

    df_sum = pd.DataFrame({
        "is_inside_grid": np.ones(len(T_all), dtype=bool),
        "has_valid_land_price": valid_lp_mask,
        "payback_years": T_all
    })

    df_sum["is_feasible"] = pd.Series([pd.NA] * len(df_sum), dtype="object")
    df_sum.loc[valid_lp_mask, "is_feasible"] = feasible_flag

    cats = classify_payback(T_all[valid_lp_mask], feasible_flag)
    df_sum["category"] = pd.Series([pd.NA] * len(df_sum), dtype="object")
    df_sum.loc[valid_lp_mask, "category"] = cats

    n_inside = len(df_sum)
    n_valid = int(valid_lp_mask.sum())
    n_invalid_lp = int((~valid_lp_mask).sum())

    n_infeasible = int((df_sum["category"] == "infeasible").sum())
    n_lt5 = int((df_sum["category"] == "<5").sum())
    n_5_20 = int((df_sum["category"] == "5-20").sum())
    n_20_50 = int((df_sum["category"] == "20-50").sum())
    n_gt50 = int((df_sum["category"] == ">50").sum())

    pct_valid = lambda n: 100.0 * n / n_valid if n_valid > 0 else np.nan
    pct_inside = lambda n: 100.0 * n / n_inside if n_inside > 0 else np.nan

    summary = {
        "type": type_name,
        "n_inside_grid": n_inside,
        "n_valid_land_price": n_valid,
        "n_invalid_land_price": n_invalid_lp,

        "n_<5": n_lt5,
        "n_5_20": n_5_20,
        "n_20_50": n_20_50,
        "n_>50": n_gt50,
        "n_infeasible": n_infeasible,

        "pct_valid_<5": pct_valid(n_lt5),
        "pct_valid_5_20": pct_valid(n_5_20),
        "pct_valid_20_50": pct_valid(n_20_50),
        "pct_valid_>50": pct_valid(n_gt50),
        "pct_valid_infeasible": pct_valid(n_infeasible),

        "pct_inside_<5": pct_inside(n_lt5),
        "pct_inside_5_20": pct_inside(n_5_20),
        "pct_inside_20_50": pct_inside(n_20_50),
        "pct_inside_>50": pct_inside(n_gt50),
        "pct_inside_infeasible": pct_inside(n_infeasible),
        "pct_inside_invalid_landprice": pct_inside(n_invalid_lp),

        "min_payback_valid": float(np.nanmin(T_all[valid_lp_mask])) if n_valid > 0 else np.nan,
        "median_payback_valid": float(np.nanmedian(T_all[valid_lp_mask])) if n_valid > 0 else np.nan,
        "max_payback_valid": float(np.nanmax(T_all[valid_lp_mask])) if n_valid > 0 else np.nan,
    }
    return df_sum, summary

dfA_grid, sumA = summarize_type(TA, valid_lp, okA, "A")
dfB_grid, sumB = summarize_type(TB, valid_lp, okB, "B")

summary_df = pd.DataFrame([sumA, sumB])

print("\n=== SUMMARY FOR TEXT ===")
print(summary_df.to_string(index=False))

print("\n=== TEXT-FRIENDLY ===")
for s in [sumA, sumB]:
    print(f"\nType {s['type']}")
    print(f"Valid grid cells: {s['n_valid_land_price']} / {s['n_inside_grid']} ({100*s['n_valid_land_price']/s['n_inside_grid']:.2f}% of inside-grid cells)")
    print(f"<5 years: {s['pct_valid_<5']:.2f}% of valid cells")
    print(f"5-20 years: {s['pct_valid_5_20']:.2f}% of valid cells")
    print(f"20-50 years: {s['pct_valid_20_50']:.2f}% of valid cells")
    print(f">50 years: {s['pct_valid_>50']:.2f}% of valid cells")
    print(f"Infeasible: {s['pct_valid_infeasible']:.2f}% of valid cells")
    print(f"Min/Median/Max payback: {s['min_payback_valid']:.2f} / {s['median_payback_valid']:.2f} / {s['max_payback_valid']:.2f}")
